In [ ]:
# Install dependencies, e.g., on Google Colab
try:
    import msaexp

except ImportError:

    ! pip install msaexp
    ! pip install git+https://github.com/karllark/dust_attenuation.git
    
    import eazy
    eazy.fetch_eazy_photoz()
    
    
    %matplotlib inline

import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

from scipy.spatial import cKDTree

import astropy.io.fits as pyfits
from astropy.utils.data import download_file
from astropy.cosmology import WMAP9
import astropy.units as u

import grizli
import grizli.catalog
from grizli import utils

import eazy
import msaexp

from astroquery.mast import Observations
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u

CACHE_DOWNLOADS = True

print(f'grizli version: {grizli.__version__}')
print(f'eazy-py version: {eazy.__version__}')
print(f'msaexp version: {msaexp.__version__}')

In [ ]:
# Full table

version = "v4.4" # Updated September 5, 2025.  Include all public spectra even without redshift / line fits

URL_PREFIX = "https://s3.amazonaws.com/msaexp-nirspec/extractions"

# Use the Zenodo release
if version == "v4.4":
    URL_PREFIX = "https://zenodo.org/records/15472354/files/"

table_url = f"{URL_PREFIX}/dja_msaexp_emission_lines_{version}.csv.gz"

tab = utils.read_catalog(download_file(table_url, cache=CACHE_DOWNLOADS), format='csv')

In [ ]:
from astropy.table import Table
import os
_paths = [
    '../catalogues/cosmos_khostovan25/specz_compilation_COSMOS_DR1.1_unique.fits',
    '../catalogues/drive-download-20260515T072221Z-3-001/COSMOS/original catalogs/specz_compilation_COSMOS_DR1.1_unique.fits',
    '../catalogues/cosmos/original catalogs/specz_compilation_COSMOS_DR1.1_unique.fits',
]
for _p in _paths:
    if os.path.isfile(_p):
        cosmos_2026 = Table.read(_p)
        print('Loaded:', _p, ' rows:', len(cosmos_2026))
        break
else:
    raise FileNotFoundError('Khostovan COSMOS catalogue not found in known locations')


In [ ]:
# Simple sky view of COSMOS (1x1 deg) with grades 2 & 3 + COSMOS 2026 flag==4
mask = np.isin(tab["grade"], [2, 3])
sub = tab[mask]

# COSMOS field center (deg) and 1x1 deg FoV
ra0, dec0 = 150.116321, 2.20583
fov = 4.0
ra_min, ra_max = ra0 - fov / 2, ra0 + fov / 2
dec_min, dec_max = dec0 - fov / 2, dec0 + fov / 2




In [ ]:


# Query JWST imaging observations
center = SkyCoord(ra0, dec0, unit="deg", frame="icrs")
obs = Observations.query_criteria(
    coordinates=center,
    radius=1.5 * u.deg,
    obs_collection="JWST",
    dataproduct_type="image",
)

# Filter to NIRCam, NIRISS, MIRI
inst_names = obs["instrument_name"]
keep = np.array([
    any(x in str(name).upper() for x in ["NIRCAM", "NIRISS", "MIRI"])
    for name in inst_names
])
obs_filt = obs[keep]

print(f"Found {len(obs_filt)} imaging observations")

# Extract STC-S footprint strings
s_regions_jwst = obs_filt["s_region"]
s_regions_jwst = [s for s in s_regions_jwst if s and "POLYGON" in s]


# Query HST imaging observations
center = SkyCoord(ra0, dec0, unit="deg", frame="icrs")
obs = Observations.query_region(
    coordinates=center,
    radius=1.5 * u.deg
)

# Filter to HST imaging
obs_hst = obs[
    (obs["obs_collection"] == "HST") & 
    (obs["dataproduct_type"] == "image")
]

# Filter to common HST imaging instruments (ACS, WFC3)
inst_names = obs_hst["instrument_name"]
keep = np.array([
    any(x in str(name).upper() for x in ["ACS", "WFC3"])
    for name in inst_names
])
obs_filt = obs_hst[keep]

print(f"Found {len(obs_filt)} HST imaging observations")

# Extract STC-S footprint strings
s_regions_hst = obs_filt["s_region"]
s_regions_hst = [s for s in s_regions_hst if s and "POLYGON" in s]

In [ ]:



cosmos_color = "orange"

fig, ax = plt.subplots(figsize=(7, 7))

# Grade 2 & 3 galaxies from DJA
sel = sub["grade"] >=2
ax.scatter(
    sub["ra"][sel],
    sub["dec"][sel],
    s=10,
    c="brown",
    alpha=0.9,
    label=f"DJA",
    edgecolors="none",
)

# COSMOS 2026 flag==4 galaxies
cosmos_sel = cosmos_2026["flag"] == 4
ax.scatter(
    cosmos_2026["ra_corrected"][cosmos_sel],
    cosmos_2026["dec_corrected"][cosmos_sel],
    s=5,
    c=cosmos_color,
    alpha=0.9,
    label="COSMOS 2026 (flag=4)",
    edgecolors="none",
    marker=".",
    zorder=0,
)


n_plotted = 0
for s_region in s_regions_hst:
    # Split by 'POLYGON' to handle multiple polygons in one string
    polygons = s_region.split('POLYGON')
    
    for poly in polygons:
        poly = poly.strip()
        if not poly:
            continue
        
        # Extract coordinate pairs
        parts = poly.split()
        try:
            coords = [float(x) for x in parts]
        except ValueError:
            continue
            
        if len(coords) % 2 != 0:
            continue
        
        # Reshape into RA, Dec pairs and close the polygon
        ra_vals = coords[::2]
        dec_vals = coords[1::2]
        ra_vals.append(ra_vals[0])
        dec_vals.append(dec_vals[0])
            
        if n_plotted==0:label='JWST NIRCam/NIRISS/MIRI'
        else: label='__None__'
        ax.plot(ra_vals, dec_vals, color="k", lw=0.15, alpha=0.8, label=label)
        n_plotted += 1
    


n_plotted = 0
for s_region in s_regions_jwst:
    # STC-S format: "POLYGON ra1 dec1 ra2 dec2 ra3 dec3 ra4 dec4"
    parts = s_region.split()
    if parts[0] != "POLYGON":
        continue
    
    # Extract coordinate pairs
    coords = [float(x) for x in parts[1:]]
    if len(coords) % 2 != 0:
        continue
    
    # Reshape into RA, Dec pairs and close the polygon
    ra_vals = coords[::2]
    dec_vals = coords[1::2]
    ra_vals.append(ra_vals[0])
    dec_vals.append(dec_vals[0])
    if n_plotted==0:label='HST ACS/WFC3'
    else: label='__None__'
    ax.plot(ra_vals, dec_vals, color="#1f77b4", lw=0.4, alpha=0.8, label=label)
    n_plotted += 1
    


ax.set_xlim(ra_max, ra_min)  # RA increases to the left on the sky
ax.set_ylim(dec_min, dec_max)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.set_title("COSMOS")
ax.grid(True, alpha=0.3, linewidth=0.6)
ax.legend(frameon=True)

plt.tight_layout()
plt.show()


## EMPOWER COSMOS spec-z diagnostics

Catalogues used:
- **Khostovan+25 DR1.1 unique** (`specz_compilation_COSMOS_DR1.1_unique.fits`, 261,975 sources) — `flag`
  follows the Khostovan QC code: 4 = highest confidence, 3 = secure, 2 = probable, 1 = uncertain, 0
  = unknown, 14/19 = stellar, 9-13 = special. `Confidence_level` is the harmonised 0–97 number.
- **DJA NIRSpec compilation** (`dja_msaexp_emission_lines_v4.4.csv.gz`) — `grade` 3 = secure, 2 =
  good, 1 = poor.
- **EMPOWER first-tier curated** (`KMOS_cosmos_uvista_first_tier.fits` etc.) — already cross-matched
  to COSMOS20 Classic/Farmer, COSMOS2025/2015/2009, UVISTA YJHK, LePhare/EAZY masses & SFRs.
- **KMOS Feb 2026 observed targets** (`cosmos_observed_11_16_02_2026_target.fits`, 24 IFUs).

EMPOWER target windows (P117 onwards, COSMOS visible December 2026):
- **Tier 1** centred z = 0.75; KARMA Hβ+Hα simultaneous IZ window **0.60 < z < 0.6325**.
- **Tier 2** centred z = 1.6 (Hα → H band).
- **Tier 3** centred z = 2.3 (Hα → K band).


In [ ]:
# === EMPOWER constants & helpers ===
import os
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib import gridspec

from astropy.table import Table, join
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

# Cosmology (Planck18-like — final z→d_C only needs h, Om consistency)
COSMO = FlatLambdaCDM(H0=70.0, Om0=0.3)

# Field centre (COSMOS)
RA0, DEC0 = 150.116321, 2.20583

# EMPOWER tier windows (centre, half-width) — half-width here is for diagnostics, not selection
TIERS = {
    "Tier 1 (z~0.75)": dict(zc=0.75, dz=0.10, color="#1f77b4"),
    "Tier 2 (z~1.6)":  dict(zc=1.60, dz=0.20, color="#2ca02c"),
    "Tier 3 (z~2.3)":  dict(zc=2.30, dz=0.20, color="#d62728"),
}

# KARMA narrow IZ window where Hβ+Hα are simultaneously visible
KARMA_IZ_HB_HA = (0.60, 0.6325)

# KMOS band ranges (microns) — approximate as published
KMOS_BANDS = {
    "IZ":  (0.78, 1.08),
    "YJ":  (1.025, 1.345),
    "H":   (1.460, 1.850),
    "HK":  (1.480, 2.450),
    "K":   (1.950, 2.450),
}

# Rest-frame wavelengths of key lines (Å)
LINES_AA = {
    "Hβ":     4861.33,
    "[OIII]4959": 4958.91,
    "[OIII]5007": 5006.84,
    "[NII]6548":  6548.05,
    "Hα":     6562.80,
    "[NII]6584":  6583.45,
}

# Major OH-bright “forbidden” λ regions in the near-IR (microns).
# These are rough envelopes — use a real OH spectrum for serious sky-line filtering.
OH_FORESTS_UM = [
    (0.86, 0.88),
    (0.92, 0.94),
    (1.08, 1.12),
    (1.18, 1.22),
    (1.27, 1.32),
    (1.42, 1.50),
    (1.55, 1.63),
    (1.73, 1.78),
    (1.83, 1.92),
]

# Paths
HERE = os.getcwd()
DROPBOX_ROOT = os.path.abspath(os.path.join(HERE, ".."))
KHOSTOVAN_FITS_CANDIDATES = [
    os.path.join(DROPBOX_ROOT, "catalogues", "cosmos_khostovan25",
                 "specz_compilation_COSMOS_DR1.1_unique.fits"),
    os.path.join(DROPBOX_ROOT, "catalogues",
                 "drive-download-20260515T072221Z-3-001", "COSMOS", "original catalogs",
                 "specz_compilation_COSMOS_DR1.1_unique.fits"),
    os.path.join(DROPBOX_ROOT, "catalogues", "cosmos", "original catalogs",
                 "specz_compilation_COSMOS_DR1.1_unique.fits"),
]

EMPOWER_COSMOS_ROOT = os.path.join(
    DROPBOX_ROOT, "catalogues", "drive-download-20260515T072221Z-3-001", "COSMOS"
)

FIG_DIR = os.path.join(HERE, "figures")
os.makedirs(FIG_DIR, exist_ok=True)


def _resolve(candidates):
    for p in candidates:
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"None of these exist:\n  " + "\n  ".join(candidates))


def tier_mask(z, name):
    cfg = TIERS[name]
    return (z >= cfg["zc"] - cfg["dz"]) & (z <= cfg["zc"] + cfg["dz"])


def line_obs_um(rest_aa, z):
    """Observed-frame λ in microns."""
    return (rest_aa / 1e4) * (1.0 + z)


print("EMPOWER constants loaded.")
print("Field centre   :", RA0, DEC0)
print("KARMA IZ Hβ+Hα:", KARMA_IZ_HB_HA)
print("Tiers          :", list(TIERS.keys()))


In [ ]:
# === Load catalogues (Khostovan unique + EMPOWER curated + observed KMOS targets) ===
khostovan_path = _resolve(KHOSTOVAN_FITS_CANDIDATES)
print("Khostovan path:", khostovan_path)
khostovan = Table.read(khostovan_path)
print(f"  rows: {len(khostovan):,}")

# Make sure the original notebook variable still resolves (cell 3 expects `cosmos_2026`).
cosmos_2026 = khostovan

# EMPOWER curated first-tier catalogues
empower_all   = Table.read(os.path.join(EMPOWER_COSMOS_ROOT,
                            "SF selection first tier", "KMOS_cosmos_all_first_tier.fits"))
empower_uvist = Table.read(os.path.join(EMPOWER_COSMOS_ROOT,
                            "SF selection first tier", "KMOS_cosmos_uvista_first_tier.fits"))
empower_h214  = Table.read(os.path.join(EMPOWER_COSMOS_ROOT,
                            "SF selection first tier", "KMOS_cosmos_uvista_H_21.4_first_tier.fits"))

# KMOS observed Feb 2026
kmos_obs_feb26 = Table.read(os.path.join(EMPOWER_COSMOS_ROOT,
                            "observed targets", "cosmos_observed_11_16_02_2026_target.fits"))

# LEGA-C bonafide RA/Dec
legac_radec = np.loadtxt(os.path.join(EMPOWER_COSMOS_ROOT, "LEGA-C",
                                       "RADEC_LEGAC_bonafide.dat"))

print(f"  EMPOWER all first-tier      : {len(empower_all):,}")
print(f"  EMPOWER UVISTA first-tier   : {len(empower_uvist):,}")
print(f"  EMPOWER UVISTA H<21.4       : {len(empower_h214):,}")
print(f"  KMOS Feb 2026 observed IFUs : {len(kmos_obs_feb26)}")
print(f"  LEGA-C bonafide             : {len(legac_radec)}")


In [ ]:
# === Tier / quality / KARMA-window counts ===
z   = khostovan["specz"]
f   = khostovan["flag"]
sec = np.isin(f, [3, 4])           # "secure or higher"
top = (f == 4)                     # highest confidence
star_like = np.isin(f, [14, 19])

print(f"Khostovan unique         : {len(khostovan):,}")
print(f"  flag=4 (top)           : {top.sum():,}")
print(f"  flag=3 or 4 (secure)   : {sec.sum():,}")
print(f"  flag in [14,19] (star) : {star_like.sum():,}")

print()
print("Counts inside EMPOWER tier windows (z within ±dz of centre):")
for name in TIERS:
    m = tier_mask(z, name) & sec
    print(f"  {name:<18s} secure : {m.sum():,} (top-only: {(m & top).sum():,})")

karma = (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])
print()
print(f"KARMA IZ Hβ+Hα ({KARMA_IZ_HB_HA[0]} < z < {KARMA_IZ_HB_HA[1]}):")
print(f"  secure : {(karma & sec).sum():,}")
print(f"  top    : {(karma & top).sum():,}")

# Survey contribution to flag=4 inside tier 1
print()
print("Top-10 surveys contributing to flag=4 inside Tier 1 + 2 + 3:")
for name in TIERS:
    m = tier_mask(z, name) & top
    surveys, counts = np.unique(khostovan["survey"][m], return_counts=True)
    order = np.argsort(counts)[::-1][:10]
    print(f"  {name}:")
    for s, c in zip(surveys[order], counts[order]):
        print(f"      survey ID {int(s):3d}: {c:,}")


In [ ]:
# === Redshift distribution with EMPOWER tiers + KARMA IZ window ===
fig, ax = plt.subplots(figsize=(11, 4.5))

bins = np.arange(0.0, 6.0, 0.02)
z = khostovan["specz"]
f = khostovan["flag"]

# Background: all sources with any redshift (≥0)
ax.hist(z[(z > 0) & ~np.isin(f, [14, 19])], bins=bins,
        histtype="stepfilled", color="0.85", edgecolor="0.7", label="Khostovan: all flags")

# Secure (flag 3+4)
sec = np.isin(f, [3, 4])
ax.hist(z[sec & (z > 0)], bins=bins,
        histtype="step", color="#444", lw=1.4, label="Khostovan: flag 3+4 (secure)")

# Top (flag=4)
top = (f == 4)
ax.hist(z[top & (z > 0)], bins=bins,
        histtype="stepfilled", color="orange", alpha=0.65, edgecolor="darkorange",
        label="Khostovan: flag=4 (highest)")

# DJA grade>=2 inside COSMOS field
try:
    sel_dja = np.isin(tab["grade"], [2, 3])
    in_cos  = (np.abs(tab["ra"] - RA0) < 1.5) & (np.abs(tab["dec"] - DEC0) < 1.5)
    ax.hist(tab["z_best"][sel_dja & in_cos & (tab["z_best"] > 0)], bins=bins,
            histtype="step", color="brown", lw=1.4, label="DJA NIRSpec (grade 2-3)")
except (NameError, KeyError):
    pass

# Tier windows
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.12)
    ax.axvline(cfg["zc"], color=cfg["color"], lw=1.2, ls="--", alpha=0.7)
    ax.text(cfg["zc"], 0.97, name, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9, color=cfg["color"])

# KARMA Hβ+Hα window
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.25)
ax.text(np.mean(KARMA_IZ_HB_HA), 0.85, "KARMA IZ\nHβ+Hα",
        transform=ax.get_xaxis_transform(),
        ha="center", va="top", fontsize=8, color="navy", weight="bold")

ax.set_xlim(0, 4.0)
ax.set_yscale("log")
ax.set_xlabel("Spectroscopic redshift")
ax.set_ylabel("N")
ax.set_title("COSMOS spec-z compilation — EMPOWER tier coverage")
ax.legend(loc="upper right", fontsize=8, frameon=True)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_zhist_tiers.png"), dpi=160)
plt.show()


In [ ]:
# === Sky maps per EMPOWER tier ===
z   = khostovan["specz"]
ra  = khostovan["ra_corrected"]
dec = khostovan["dec_corrected"]
top = (khostovan["flag"] == 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
for ax, name in zip(axes, TIERS):
    cfg = TIERS[name]
    m = top & tier_mask(z, name)

    ax.scatter(ra[m], dec[m], s=4, c=cfg["color"], alpha=0.55,
               edgecolors="none", label=f"Khostovan flag=4 ({m.sum():,})")

    # LEGA-C bonafide
    ax.scatter(legac_radec[:, 0], legac_radec[:, 1], s=10,
               facecolors="none", edgecolors="k", lw=0.5, label="LEGA-C bonafide")

    # KMOS observed Feb 2026 (only physically in tier 1 z-window)
    if "Tier 1" in name:
        ax.scatter(kmos_obs_feb26["ra"], kmos_obs_feb26["dec"], s=80,
                   marker="*", facecolors="gold", edgecolors="k", lw=0.6,
                   label="KMOS observed (Feb 2026)")

    # KARMA Hβ+Hα subset highlighted in tier 1
    if "Tier 1" in name:
        karma = top & (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])
        ax.scatter(ra[karma], dec[karma], s=14, marker="x", c="navy", lw=0.5,
                   label=f"IZ Hβ+Hα ({karma.sum():,})")

    ax.set_aspect("equal")
    ax.set_xlim(RA0 + 1.2, RA0 - 1.2)
    ax.set_ylim(DEC0 - 1.2, DEC0 + 1.2)
    ax.set_xlabel("RA [deg]")
    ax.set_title(name)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=7, frameon=True)

axes[0].set_ylabel("Dec [deg]")
fig.suptitle("COSMOS sky maps per EMPOWER tier (flag=4 only)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_sky_per_tier.png"), dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# === RA-z and Dec-z wedge plots (LSS view) ===
z   = khostovan["specz"]
ra  = khostovan["ra_corrected"]
dec = khostovan["dec_corrected"]
top = (khostovan["flag"] == 4)

zmax = 3.0
m = top & (z > 0) & (z < zmax)

fig, axes = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True)
for ax, axis, lbl in zip(axes, [ra, dec], ["RA [deg]", "Dec [deg]"]):
    ax.scatter(z[m], axis[m], s=1.5, c="0.25", alpha=0.35, edgecolors="none")

    # Tier windows
    for name, cfg in TIERS.items():
        ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
                   color=cfg["color"], alpha=0.12)
        ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)

    # KARMA IZ window
    ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.25)

    ax.set_ylabel(lbl)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Spectroscopic redshift")
axes[0].set_title("COSMOS flag=4 large-scale structure (RA-z and Dec-z)")
axes[0].set_xlim(0, zmax)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_wedge_lss.png"), dpi=160)
plt.show()


In [ ]:
# === 3D comoving Cartesian view of the Tier 1 (0.55<z<0.95) slice ===
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers projection)

z   = khostovan["specz"]
ra  = khostovan["ra_corrected"]
dec = khostovan["dec_corrected"]
top = (khostovan["flag"] == 4)

zmin, zmax = 0.55, 0.95
m = top & (z > zmin) & (z < zmax)

# Comoving distance and tangential offsets relative to field centre
d_C = COSMO.comoving_distance(z[m]).to(u.Mpc).value
ra_off  = (ra[m]  - RA0)  * np.cos(np.deg2rad(DEC0))    # deg
dec_off = (dec[m] - DEC0)                               # deg
# Convert angular to transverse comoving (small-angle, in cMpc)
x_cMpc = np.deg2rad(ra_off)  * d_C
y_cMpc = np.deg2rad(dec_off) * d_C
zc_box = d_C - np.median(d_C)

fig = plt.figure(figsize=(12, 5.2))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.scatter(x_cMpc, y_cMpc, zc_box, s=2.5, c=z[m], cmap="viridis", alpha=0.5)
ax3d.set_xlabel("x [cMpc]")
ax3d.set_ylabel("y [cMpc]")
ax3d.set_zlabel("d_C - <d_C> [cMpc]")
ax3d.set_title(f"Tier 1 slice 3D ({zmin} < z < {zmax}, N={m.sum():,})")

# Project onto x-z to show the wall structure (COSMOS Wall ≈ z~0.73)
ax2 = fig.add_subplot(1, 2, 2)
sc = ax2.scatter(x_cMpc, zc_box, s=3.5, c=z[m], cmap="viridis", alpha=0.6,
                 edgecolors="none")
plt.colorbar(sc, ax=ax2, label="z")
ax2.set_xlabel("x [cMpc]  (RA direction)")
ax2.set_ylabel("d_C - <d_C> [cMpc]")
ax2.set_title("Top-down (x, d_C) projection")
ax2.set_aspect("equal", adjustable="datalim")
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "04_3d_tier1.png"), dpi=160)
plt.show()


In [ ]:
# === Mass / H-band funnel from the EMPOWER curated first-tier ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# (a) log M* vs z
ax = axes[0]
m_finite = np.isfinite(empower_all["lp_mass_best"]) & np.isfinite(empower_all["specz"])
ax.scatter(empower_all["specz"][m_finite], empower_all["lp_mass_best"][m_finite],
           s=4, c="0.55", alpha=0.4, label=f"first-tier all (N={m_finite.sum():,})")

uvist_m = np.isfinite(empower_uvist["lp_mass_best"]) & np.isfinite(empower_uvist["specz"])
ax.scatter(empower_uvist["specz"][uvist_m], empower_uvist["lp_mass_best"][uvist_m],
           s=6, c="#1f77b4", alpha=0.6, label=f"UVISTA first-tier (N={uvist_m.sum():,})")

h214_m = np.isfinite(empower_h214["lp_mass_best"]) & np.isfinite(empower_h214["specz"])
ax.scatter(empower_h214["specz"][h214_m], empower_h214["lp_mass_best"][h214_m],
           s=10, c="orange", alpha=0.8, edgecolors="k", lw=0.2,
           label=f"UVISTA H<21.4 (N={h214_m.sum():,})")

ax.axhline(10.0, ls="--", c="r", label="log M* = 10 (EMPOWER floor)")
ax.set_xlabel("specz")
ax.set_ylabel("log M* / M⊙  (LePhare best)")
ax.set_title("Mass funnel")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

# (b) H-band magnitude vs z
ax = axes[1]
h = empower_all["UVISTA_H_MAG_AUTO"]
ok = np.isfinite(h) & np.isfinite(empower_all["specz"]) & (h > 14) & (h < 30)
ax.scatter(empower_all["specz"][ok], h[ok], s=4, c="0.55", alpha=0.4, label="first-tier all")

h2 = empower_h214["UVISTA_H_MAG_AUTO"]
ok2 = np.isfinite(h2) & (h2 < 30)
ax.scatter(empower_h214["specz"][ok2], h2[ok2], s=10, c="orange", alpha=0.85,
           edgecolors="k", lw=0.2, label="UVISTA H<21.4 (selected)")

ax.axhline(21.4, ls="--", c="r", label="H = 21.4")
ax.invert_yaxis()
ax.set_xlabel("specz")
ax.set_ylabel("UVISTA H_AUTO [mag]")
ax.set_title("H-band funnel")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_mass_hmag.png"), dpi=160)
plt.show()


In [ ]:
# === KARMA observability atlas: λ_obs vs z for the EMPOWER lines, KMOS bands, OH bands ===
zgrid = np.linspace(0.0, 3.0, 600)
fig, ax = plt.subplots(figsize=(11.5, 5.5))

# OH-forest envelopes (background)
for lo, hi in OH_FORESTS_UM:
    ax.axhspan(lo, hi, color="0.85", alpha=0.5, zorder=0)

# KMOS band envelopes
band_colors = {"IZ": "#1f77b4", "YJ": "#2ca02c", "H": "#d62728",
               "HK": "#9467bd", "K": "#8c564b"}
for band, (lo, hi) in KMOS_BANDS.items():
    ax.axhspan(lo, hi, color=band_colors[band], alpha=0.10, zorder=0)
    ax.text(3.02, 0.5 * (lo + hi), band, fontsize=9, color=band_colors[band],
            weight="bold", va="center")

# Tier windows
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)

ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.25)

# Lines
line_styles = {"Hβ": "-", "[OIII]5007": "--", "Hα": "-", "[NII]6584": ":"}
line_colors = {"Hβ": "C0", "[OIII]5007": "C2", "Hα": "C3", "[NII]6584": "C1"}
for ln in ["Hβ", "[OIII]5007", "Hα", "[NII]6584"]:
    ax.plot(zgrid, line_obs_um(LINES_AA[ln], zgrid),
            ls=line_styles[ln], color=line_colors[ln], lw=1.6, label=ln)

ax.set_xlim(0, 3.0)
ax.set_ylim(0.7, 2.6)
ax.set_xlabel("Redshift")
ax.set_ylabel("Observed-frame λ [µm]")
ax.set_title("KARMA observability: KMOS bands (coloured), OH-forest envelopes (grey), "
             "EMPOWER tier windows (vertical)")
ax.legend(loc="upper left", fontsize=9, frameon=True)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_karma_atlas.png"), dpi=160)
plt.show()


In [ ]:
# === Spec-z surface density per tier (proxy for clusters/filaments/field) ===
# Note: the Khostovan unique table's GroupID/GroupSize column tracks duplicate
# spec-z measurements per source, not cosmological FoF groups. For true
# group/environment classification we need the Knobel+12 zCOSMOS FoF and
# Toni+25 AMICO COSMOS-Web catalogues (not yet ingested).
from scipy.stats import gaussian_kde

z   = khostovan["specz"]
ra  = khostovan["ra_corrected"]
dec = khostovan["dec_corrected"]
top = (khostovan["flag"] == 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
for ax, name in zip(axes, TIERS):
    cfg = TIERS[name]
    m = top & tier_mask(z, name)
    if m.sum() < 50:
        ax.set_title(f"{name}: not enough sources ({m.sum()})")
        continue

    # 2D hexbin density (per square arcmin equivalent)
    hb = ax.hexbin(ra[m], dec[m], gridsize=60,
                   extent=[RA0 - 1.2, RA0 + 1.2, DEC0 - 1.2, DEC0 + 1.2],
                   mincnt=1, cmap="inferno")
    cb = plt.colorbar(hb, ax=ax, label="N flag=4 per hex")

    # Overdensity ridge: identify hexes >2σ above field mean
    counts = hb.get_array()
    if counts.size:
        thr = np.mean(counts) + 2.0 * np.std(counts)
        ax.text(0.02, 0.98, f"N={m.sum():,}\n>2σ thr={thr:.1f}",
                transform=ax.transAxes, va="top", fontsize=8,
                color="white", bbox=dict(facecolor="black", alpha=0.5))

    # Tier 1: KARMA Hβ+Hα subset
    if "Tier 1" in name:
        karma = top & (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])
        ax.scatter(ra[karma], dec[karma], s=6, marker="x", c="cyan", lw=0.4,
                   label=f"IZ Hβ+Hα ({karma.sum():,})")
        ax.scatter(kmos_obs_feb26["ra"], kmos_obs_feb26["dec"], s=80,
                   marker="*", facecolors="yellow", edgecolors="k", lw=0.6,
                   label="KMOS Feb 2026")
        ax.legend(loc="upper left", fontsize=7, frameon=True)

    ax.set_aspect("equal")
    ax.set_xlim(RA0 + 1.2, RA0 - 1.2)
    ax.set_ylim(DEC0 - 1.2, DEC0 + 1.2)
    ax.set_xlabel("RA [deg]")
    ax.set_title(name)

axes[0].set_ylabel("Dec [deg]")
fig.suptitle("Spec-z flag=4 surface density per tier (overdensities ≈ candidate clusters/filaments)",
             y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_density_per_tier.png"), dpi=160, bbox_inches="tight")
plt.show()

# Print N and median density per tier for the proposal
print()
print("Spec-z surface density summary (flag=4, COSMOS 2.4x2.4 deg box):")
area_deg2 = 2.4 * 2.4
for name in TIERS:
    m = top & tier_mask(z, name)
    in_box = m & (np.abs(ra - RA0) < 1.2) & (np.abs(dec - DEC0) < 1.2)
    print(f"  {name:<22s} N(in 2.4x2.4 deg box) = {in_box.sum():,}  "
          f"=> {in_box.sum()/area_deg2:.0f} /deg^2")


In [ ]:
# === Khostovan flag=4 vs EMPOWER curated first-tier overlap ===
khost_sec = khostovan[(khostovan["flag"] == 4)
                      & tier_mask(khostovan["specz"], "Tier 1 (z~0.75)")]
c_khost = SkyCoord(khost_sec["ra_corrected"], khost_sec["dec_corrected"], unit="deg")
c_emp   = SkyCoord(empower_all["ALPHA_J2000"], empower_all["DELTA_J2000"], unit="deg")
idx, sep, _ = c_emp.match_to_catalog_sky(c_khost)
ok = sep.arcsec < 1.0
arcsec_label = "1 arcsec"
print(f"EMPOWER first-tier rows           : {len(empower_all):,}")
print(f"  with Khostovan flag=4 ({arcsec_label}) : {ok.sum():,}")
print(f"  match fraction                  : {ok.mean():.2%}")

# (a) separation histogram
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(sep.arcsec[sep.arcsec < 5.0], bins=80, color="0.5", edgecolor="k")
ax.axvline(1.0, color="r", ls="--", label=arcsec_label)
ax.set_xlabel("Nearest Khostovan flag=4 match  [arcsec]")
ax.set_ylabel("N (EMPOWER curated)")
ax.set_title("Crossmatch separation distribution")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "08_xmatch_sep.png"), dpi=160)
plt.show()

# (b) mass distribution: matched vs unmatched
fig, ax = plt.subplots(figsize=(7, 4.2))
m_finite = np.isfinite(empower_all["lp_mass_best"])
bins = np.arange(8.0, 12.0, 0.1)
ax.hist(empower_all["lp_mass_best"][m_finite &  ok], bins=bins, alpha=0.6,
        color="#1f77b4", label="matched (flag=4 within 1\")")
ax.hist(empower_all["lp_mass_best"][m_finite & ~ok], bins=bins, alpha=0.6,
        color="0.55", label="unmatched")
ax.axvline(10.0, ls="--", c="r")
ax.set_xlabel("log M* / M⊙  (LePhare best)")
ax.set_ylabel("N")
ax.set_title("Mass distribution of curated first-tier (matched vs unmatched to flag=4)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "09_xmatch_mass.png"), dpi=160)
plt.show()


In [ ]:
# === Summary print: what to feed the proposal / WG ===
print("=" * 72)
print(" EMPOWER COSMOS spec-z diagnostics — quick summary")
print("=" * 72)

z = khostovan["specz"]; f = khostovan["flag"]
for name in TIERS:
    m_all = tier_mask(z, name)
    m_top = m_all & (f == 4)
    print(f"  {name:<22s} flag=4: {m_top.sum():>7,}  "
          f"any flag>=3: {(m_all & np.isin(f, [3,4])).sum():>7,}")

karma = (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])
print()
print(f"  KARMA IZ Hβ+Hα (0.60<z<0.6325):")
print(f"      flag=4         : {(karma & (f == 4)).sum():,}")
print(f"      flag>=3        : {(karma & np.isin(f, [3, 4])).sum():,}")
print(f"      flag>=3 & H<21.4: see overlap with empower_h214")
print()
print(f"  Figures written to: {FIG_DIR}")
print(f"  EMPOWER curated   : {len(empower_all):,} (all), {len(empower_uvist):,} (UVISTA), "
      f"{len(empower_h214):,} (UVISTA H<21.4)")
print(f"  KMOS observed Feb 2026: {len(kmos_obs_feb26)} IFUs")


## DJA NIRSpec ↔ Khostovan+25 COSMOS comparison

Goal: identify where Khostovan DR1.1 is missing spectroscopic redshifts that exist in the
DJA NIRSpec v4.4 compilation. For each DJA source in the COSMOS field (±1.5° box) we ask:

1. Does it have *any* Khostovan entry within 1″? (any `flag`, including the 0 = unknown
   and -1/-2/-99 = no redshift buckets).
2. If yes, how secure is the Khostovan entry — i.e., is it just an unflagged duplicate
   target acquisition or a real measurement?
3. If yes and Khostovan also has a redshift, do the two agree? (Δz / (1+z) outliers
   highlight catastrophic disagreement candidates.)

Match radius: **0.5 arcsec**. Outlier threshold: **|Δz/(1+z)| > 0.01**.


In [ ]:
# === DJA NIRSpec v4.4 vs Khostovan DR1.1 — gap analysis ===
# Prerequisites: cell 1 (DJA `tab`) and cells empower-01/02 (`khostovan`, FIG_DIR) must run first.
assert "tab" in dir(), "Run cell 1 first to load the DJA table (`tab`)."
assert "khostovan" in dir(), "Run cell empower-02 first to load Khostovan."

MATCH_RADIUS_ARCSEC = 0.5
DZ_OUTLIER_THR = 0.01      # |dz/(1+z)| above this counts as catastrophic
COSMOS_HALF_DEG = 1.5      # square half-side for COSMOS box

# --- DJA inside COSMOS box ---
dja_in = ((np.abs(tab["ra"] - RA0) < COSMOS_HALF_DEG)
          & (np.abs(tab["dec"] - DEC0) < COSMOS_HALF_DEG))
dja = tab[dja_in]
print(f"DJA v4.4 inside COSMOS  ({COSMOS_HALF_DEG}° half-side box): {len(dja):,}")

# Grade breakdown
for g in [1, 2, 3]:
    n = (dja["grade"] == g).sum()
    print(f"   grade {g}: {n:,}")
n_nograde = (~np.isin(dja["grade"], [1, 2, 3])).sum()
print(f"   grade other/none: {n_nograde:,}")

# --- Crossmatch DJA → Khostovan unique (any flag) ---
c_dja   = SkyCoord(dja["ra"], dja["dec"], unit="deg")
c_khost = SkyCoord(khostovan["ra_corrected"], khostovan["dec_corrected"], unit="deg")
idx, sep, _ = c_dja.match_to_catalog_sky(c_khost)
matched = sep.arcsec < MATCH_RADIUS_ARCSEC

print()
print(f"DJA sources with ANY Khostovan entry within {MATCH_RADIUS_ARCSEC}\":  "
      f"{matched.sum():,} / {len(dja):,}  ({matched.mean():.1%})")
print(f"DJA sources MISSING from Khostovan                : "
      f"{(~matched).sum():,}  ({(~matched).mean():.1%})")

# Khostovan flag/spec-z arrays aligned to dja rows (only valid where matched)
khost_flag = khostovan["flag"][idx]
khost_z    = khostovan["specz"][idx]
khost_has_z = matched & (khost_z > 0)       # Khostovan entry actually carries a redshift
khost_secure = matched & np.isin(khost_flag, [3, 4])

# --- Per-DJA-grade matched / unmatched / matched-but-low-quality breakdown ---
print()
print("Per-DJA-grade matrix:")
print(f"  {'':<10s}{'N_DJA':>8s}{'matched':>10s}{'mat+z>0':>10s}{'mat+secure':>12s}"
      f"{'MISSING':>10s}")
for g in [3, 2, 1]:
    m_g = (dja["grade"] == g)
    n   = m_g.sum()
    n_m = (m_g & matched).sum()
    n_z = (m_g & khost_has_z).sum()
    n_s = (m_g & khost_secure).sum()
    n_u = (m_g & ~matched).sum()
    print(f"  grade {g} : {n:>8,}{n_m:>10,}{n_z:>10,}{n_s:>12,}{n_u:>10,}")

# --- Per-EMPOWER-tier slice ---
print()
print("Per EMPOWER tier (using DJA z):")
for name in TIERS:
    m_t = tier_mask(dja["z_best"], name)
    n   = m_t.sum()
    n_m = (m_t & matched).sum()
    n_s = (m_t & khost_secure).sum()
    n_u = (m_t & ~matched).sum()
    print(f"  {name:<22s} N_DJA={n:>6,}  matched={n_m:>6,}  "
          f"secure-in-Khost={n_s:>6,}  MISSING-from-Khost={n_u:>6,}")

# Tier 1 KARMA window
karma_dja = (dja["z_best"] > KARMA_IZ_HB_HA[0]) & (dja["z_best"] < KARMA_IZ_HB_HA[1])
print(f"\nKARMA IZ Hβ+Hα window: DJA N={karma_dja.sum():,}, "
      f"matched={(karma_dja & matched).sum()}, "
      f"missing={(karma_dja & ~matched).sum()}")


In [ ]:
# === DJA-vs-Khostovan diagnostic plots ===
# (uses arrays defined in the previous cell: `dja`, `matched`, `khost_flag`, `khost_z`, `sep`)

# --- (A) Stacked bar: per-grade matched vs missing, plus secondary panel showing
#         Khostovan-flag distribution of *matched* DJA sources per grade ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
grades = [3, 2, 1]
n_match = np.array([(dja["grade"] == g).sum() for g in grades])
n_in    = np.array([((dja["grade"] == g) & matched).sum() for g in grades])
n_miss  = n_match - n_in

xs = np.arange(len(grades))
ax.bar(xs, n_in,   width=0.65, color="#1f77b4", label="In Khostovan (any flag, 1\")")
ax.bar(xs, n_miss, width=0.65, bottom=n_in, color="#d62728",
       label="Missing from Khostovan")
for i, (a, b) in enumerate(zip(n_in, n_miss)):
    ax.text(xs[i], a + b + max(n_match)*0.01,
            f"{a:,} + {b:,}", ha="center", fontsize=8)
ax.set_xticks(xs)
ax.set_xticklabels([f"DJA grade {g}" for g in grades])
ax.set_ylabel("N (in COSMOS box)")
ax.set_title("DJA NIRSpec sources inside COSMOS — present vs missing in Khostovan")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.25)

# Khostovan flag distribution of matched DJA grade-3 sources
ax = axes[1]
m_g3 = (dja["grade"] == 3) & matched
if m_g3.sum():
    flags, counts = np.unique(khost_flag[m_g3], return_counts=True)
    bars = ax.bar(np.arange(len(flags)), counts, color="#2ca02c", edgecolor="k", lw=0.4)
    ax.set_xticks(np.arange(len(flags)))
    ax.set_xticklabels([str(int(f)) for f in flags], rotation=0)
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width()/2, b.get_height(),
                f"{c}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Khostovan flag")
ax.set_ylabel(f"N (matched DJA grade=3, sep<1\")")
ax.set_title("Quality of the Khostovan entry that matches DJA grade=3")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "10_dja_vs_khost_counts.png"), dpi=160)
plt.show()


In [ ]:
# --- (B) Redshift distribution of DJA-only sources (no Khostovan within 1") ---
fig, ax = plt.subplots(figsize=(11, 4.4))
bins = np.arange(0.0, 12.0, 0.05)

z_dja = dja["z_best"]
for g, c in zip([3, 2, 1], ["#2ca02c", "#1f77b4", "0.6"]):
    m = (dja["grade"] == g) & (~matched) & np.isfinite(z_dja)
    if m.sum():
        ax.hist(z_dja[m], bins=bins, histtype="step", lw=1.4, color=c,
                label=f"DJA grade {g} not in Khostovan (N={m.sum():,})")

# Tier shading
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)

ax.set_xlabel("Redshift (DJA)")
ax.set_ylabel("N")
ax.set_title("Redshift distribution of DJA sources MISSING from Khostovan")
ax.set_xlim(0, 10.0)
ax.set_yscale("log")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "11_dja_only_zdist.png"), dpi=160)
plt.show()


In [ ]:
# --- (C) Sky map: where are the DJA-only secure sources? ---
fig, ax = plt.subplots(figsize=(8, 7.2))

# Background: Khostovan flag=4 (low alpha)
top_k = (khostovan["flag"] == 4)
ax.scatter(khostovan["ra_corrected"][top_k], khostovan["dec_corrected"][top_k],
           s=2, c="0.78", alpha=0.4, label=f"Khostovan flag=4 (N={top_k.sum():,})",
           edgecolors="none")

# DJA-only by grade
for g, c, s in zip([3, 2, 1], ["#2ca02c", "#1f77b4", "#d62728"], [16, 10, 5]):
    m = (dja["grade"] == g) & (~matched)
    if m.sum():
        ax.scatter(dja["ra"][m], dja["dec"][m], s=s, c=c, alpha=0.9,
                   edgecolors="k", lw=0.2,
                   label=f"DJA grade {g}, NOT in Khostovan (N={m.sum():,})")

ax.set_aspect("equal")
ax.set_xlim(RA0 + 1.5, RA0 - 1.5)
ax.set_ylim(DEC0 - 1.5, DEC0 + 1.5)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("Where Khostovan is missing entries: DJA-only sources on the sky")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "12_dja_only_sky.png"), dpi=160)
plt.show()


In [ ]:
# --- (D) z-agreement for matched pairs: Δz/(1+z) histogram + scatter ---
both = matched & (khost_z > 0) & np.isfinite(dja["z_best"])
zd = dja["z_best"][both]
zk = khost_z[both]
dz = (zd - zk) / (1.0 + zk)
n_pairs = both.sum()
n_out = (np.abs(dz) > DZ_OUTLIER_THR).sum()

print(f"Matched pairs with both z > 0     : {n_pairs:,}")
print(f"   median Δz/(1+z)                : {np.median(dz):+.4f}")
print(f"   NMAD                           : {1.4826 * np.median(np.abs(dz - np.median(dz))):.4f}")
print(f"   catastrophic |Δz/(1+z)| > {DZ_OUTLIER_THR}: {n_out:,}  ({n_out/n_pairs:.2%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Histogram
ax = axes[0]
bins = np.linspace(-0.05, 0.05, 121)
ax.hist(np.clip(dz, bins[0], bins[-1]), bins=bins, color="#1f77b4",
        edgecolor="k", lw=0.3)
ax.axvline( DZ_OUTLIER_THR, ls="--", c="r", label=f"|Δz/(1+z)| = {DZ_OUTLIER_THR}")
ax.axvline(-DZ_OUTLIER_THR, ls="--", c="r")
ax.set_xlabel("(z_DJA − z_Khost) / (1 + z_Khost)")
ax.set_ylabel("N pairs")
ax.set_title(f"z agreement between matched DJA & Khostovan ({n_pairs:,} pairs)")
ax.legend(fontsize=8)
ax.set_yscale("log")
ax.grid(alpha=0.25)

# Scatter z_DJA vs z_Khost, colour by |dz|
ax = axes[1]
out_mask = np.abs(dz) > DZ_OUTLIER_THR
ax.scatter(zk[~out_mask], zd[~out_mask], s=2.5, c="0.5", alpha=0.45,
           edgecolors="none", label=f"agree (N={(~out_mask).sum():,})")
ax.scatter(zk[out_mask], zd[out_mask], s=6, c="r", alpha=0.85,
           edgecolors="k", lw=0.2,
           label=f"|Δz/(1+z)|>{DZ_OUTLIER_THR}  (N={out_mask.sum():,})")
zmx = max(np.nanmax(zd), np.nanmax(zk), 4.0)
ax.plot([0, zmx], [0, zmx], ls=":", c="k", lw=1.0)
ax.set_xlabel("z (Khostovan)")
ax.set_ylabel("z (DJA)")
ax.set_xlim(0, zmx)
ax.set_ylim(0, zmx)
ax.set_aspect("equal", adjustable="box")
ax.set_title("z_DJA vs z_Khost  (red = catastrophic)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "13_z_agreement.png"), dpi=160)
plt.show()


In [ ]:
# --- (E) Catastrophic-z rate split by Khostovan flag ---
# This shows that the x% global "catastrophic" rate is driven by chance matches
# to LOW-flag Khostovan entries (flag 0/1), not by real disagreements.
both = matched & (khost_z > 0) & np.isfinite(dja["z_best"]) & (dja["z_best"] > 0)
flag_arr = khost_flag[both]
dz_arr   = (dja["z_best"][both] - khost_z[both]) / (1.0 + khost_z[both])

print(f"{'Khost flag':<12s}{'N pairs':>10s}{'|Δz/(1+z)|>0.01':>20s}{'outlier %':>12s}{'NMAD':>10s}")
print("-" * 64)
for fl in [4, 3, 2, 1, 0]:
    sel = (flag_arr == fl)
    n   = sel.sum()
    if n == 0:
        print(f"  flag={fl:<6d}{n:>10,}     -- no matches --")
        continue
    n_out = (np.abs(dz_arr[sel]) > DZ_OUTLIER_THR).sum()
    nmad  = 1.4826 * np.median(np.abs(dz_arr[sel] - np.median(dz_arr[sel])))
    print(f"  flag={fl:<6d}{n:>10,}{n_out:>20,}{n_out/max(n,1)*100:>11.1f}%"
          f"{nmad:>10.4f}")

# Plot: stacked outlier breakdown
fig, ax = plt.subplots(figsize=(7, 4.4))
flags_plot = [4, 3, 2, 1, 0]
n_agree, n_outl = [], []
for fl in flags_plot:
    sel = (flag_arr == fl)
    o = (sel & (np.abs(dz_arr) > DZ_OUTLIER_THR)).sum()
    a = sel.sum() - o
    n_agree.append(a); n_outl.append(o)
xs = np.arange(len(flags_plot))
ax.bar(xs, n_agree, color="#1f77b4", label="|Δz/(1+z)| ≤ 0.01 (agree)")
ax.bar(xs, n_outl, bottom=n_agree, color="#d62728",
       label="|Δz/(1+z)| > 0.01 (catastrophic)")
ax.set_xticks(xs); ax.set_xticklabels([str(f) for f in flags_plot])
ax.set_xlabel("Khostovan flag of the matched entry")
ax.set_ylabel("N matched pairs")
ax.set_title("DJA ↔ Khostovan z-agreement, split by Khostovan flag")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.25)
for i, (a, o) in enumerate(zip(n_agree, n_outl)):
    tot = a + o
    if tot > 0:
        ax.text(xs[i], tot, f"{o/tot*100:.0f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "14_dz_per_khost_flag.png"), dpi=160)
plt.show()

# Final headline summary
print()
print("=" * 70)
print(" Headline — DJA NIRSpec gap in Khostovan (COSMOS box, 1.5° half-side)")
print("=" * 70)
grade3 = (dja["grade"] == 3)
print(f"  DJA secure (grade=3) inside COSMOS : {grade3.sum():,}")
print(f"    → present in Khostovan @ 1\": {(grade3 & matched).sum():,}")
print(f"        of which secure (flag>=3): {(grade3 & matched & np.isin(khost_flag, [3,4])).sum():,}")
print(f"    → MISSING from Khostovan       : {(grade3 & ~matched).sum():,}  "
      f"({(grade3 & ~matched).sum()/grade3.sum():.0%})")
print()
print("  Of *matched* pairs, catastrophic outlier rate (|Δz/(1+z)|>0.01):")
m_secure = (flag_arr >= 3)
n_sec = m_secure.sum()
if n_sec:
    out_sec = (m_secure & (np.abs(dz_arr) > DZ_OUTLIER_THR)).sum()
    print(f"    Khostovan flag>=3 : {out_sec:,} / {n_sec:,}  "
          f"({out_sec/n_sec:.2%})  ← real disagreement")
m_low = (flag_arr <= 1) & (flag_arr >= 0)
n_low = m_low.sum()
if n_low:
    out_low = (m_low & (np.abs(dz_arr) > DZ_OUTLIER_THR)).sum()
    print(f"    Khostovan flag<=1 : {out_low:,} / {n_low:,}  "
          f"({out_low/n_low:.2%})  ← chance matches to unreliable Khostovan entries")


## DEVILS DR1 (AAT/AAOmega) vs Khostovan+25 — D10 (COSMOS)

Pulled from DataCentral TAP service (`devils_dr1.D10MasterRedshifts`, filtered to rows with
a real DEVILS spectroscopic redshift: `DEVILS_z > 0`). Saved to
`spec_catalogues/devils_dr1/devils_dr1_d10_devils_specz.fits` (9,922 rows).

Key DEVILS columns we use:
- `RAcen`, `DECcen` — ProFound segment centre (VISTA Y-band imaging).
- `DEVILS_z` — DEVILS spectroscopic redshift.
- `DEVILS_prob` — probability from DEVILS pipeline (0–1; ≥0.8 ≈ secure).
- `starFlag`, `artefactFlag`, `mask` — quality flags.
- `mag_Y` — VISTA Y-band magnitude (DEVILS selection Y < 21.2).
- `zBestSource` — which survey is the "best" z in the DEVILS compilation (when DEVILS, it
  means DEVILS supersedes any prior literature z for that source).


In [ ]:
# === Load DEVILS DR1 D10 (COSMOS) spec-z table ===
DEVILS_PATH = os.path.join(DROPBOX_ROOT, "spec_catalogues", "devils_dr1",
                           "devils_dr1_d10_devils_specz.fits")
devils = Table.read(DEVILS_PATH)
print(f"DEVILS rows: {len(devils):,}")

# Galaxy-like, non-artefact, non-masked
devils_clean = devils[(devils["starFlag"] == 0)
                      & (devils["artefactFlag"] == 0)
                      & (devils["mask"] == 0)
                      & (devils["DEVILS_z"] > 0)]
print(f"   galaxy + clean: {len(devils_clean):,}")
print(f"   prob >= 0.8   : {(devils_clean['DEVILS_prob'] >= 0.8).sum():,}")
print(f"   prob >= 0.95  : {(devils_clean['DEVILS_prob'] >= 0.95).sum():,}")

# Per-tier (DEVILS_prob >= 0.8)
print()
print("DEVILS prob>=0.8 per EMPOWER tier:")
sel = devils_clean["DEVILS_prob"] >= 0.8
for name in TIERS:
    m = sel & tier_mask(devils_clean["DEVILS_z"], name)
    print(f"  {name:<22s} {m.sum():>5,}")
karma = sel & (devils_clean["DEVILS_z"] > KARMA_IZ_HB_HA[0]) & (devils_clean["DEVILS_z"] < KARMA_IZ_HB_HA[1])
print(f"  KARMA IZ Hβ+Hα         {karma.sum():,}")

# zBestSource summary — when does DEVILS supersede existing z?
print()
print("Where DEVILS is the 'best' redshift (zBestSource):")
src, cnt = np.unique(devils_clean["zBestSource"], return_counts=True)
for s, c in sorted(zip(src, cnt), key=lambda x: -x[1])[:8]:
    print(f"  {s:>14s}: {c:,}")


In [ ]:
# === DEVILS D10 sky coverage over COSMOS ===
fig, ax = plt.subplots(figsize=(8, 7.4))

# Khostovan flag=4 in the background
top_k = (khostovan["flag"] == 4)
ax.scatter(khostovan["ra_corrected"][top_k], khostovan["dec_corrected"][top_k],
           s=1.5, c="0.78", alpha=0.35, edgecolors="none",
           label=f"Khostovan flag=4 (N={top_k.sum():,})")

# DEVILS sources, colour-coded by prob bin
clean = devils_clean
p_bins = [(0.0, 0.5, "0.6", "prob<0.5"),
          (0.5, 0.8, "#ff7f0e", "0.5≤prob<0.8"),
          (0.8, 0.95, "#1f77b4", "0.8≤prob<0.95"),
          (0.95, 1.01, "#2ca02c", "prob≥0.95")]
for lo, hi, c, lbl in p_bins:
    m = (clean["DEVILS_prob"] >= lo) & (clean["DEVILS_prob"] < hi)
    if m.sum():
        ax.scatter(clean["RAcen"][m], clean["DECcen"][m], s=3, c=c,
                   alpha=0.7, edgecolors="none", label=f"DEVILS {lbl} ({m.sum():,})")

# Approximate DEVILS D10 footprint (RA 149.38–150.70, Dec 1.65–2.79)
ra_lo, ra_hi = np.nanmin(clean["RAcen"]), np.nanmax(clean["RAcen"])
de_lo, de_hi = np.nanmin(clean["DECcen"]), np.nanmax(clean["DECcen"])
ax.plot([ra_lo, ra_hi, ra_hi, ra_lo, ra_lo],
        [de_lo, de_lo, de_hi, de_hi, de_lo],
        "k--", lw=0.8, label="D10 footprint (bbox)")

ax.set_aspect("equal")
ax.set_xlim(RA0 + 1.2, RA0 - 1.2)
ax.set_ylim(DEC0 - 1.2, DEC0 + 1.2)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("DEVILS DR1 D10 spec-z over COSMOS")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "15_devils_sky.png"), dpi=160)
plt.show()

# Approx D10 area (cos-corrected)
area = (ra_hi - ra_lo) * (de_hi - de_lo) * np.cos(np.deg2rad(0.5*(de_lo + de_hi)))
print(f"DEVILS D10 bounding box: RA [{ra_lo:.3f}, {ra_hi:.3f}], "
      f"Dec [{de_lo:.3f}, {de_hi:.3f}]")
print(f"   ≈ {area:.3f} deg² (cos-corrected; actual coverage is a 1×1.5° rectangle)")


In [ ]:
# === Crossmatch DEVILS DR1 → Khostovan+25 ===
# For each DEVILS spec-z, find its nearest Khostovan entry within 0.5″.
DEVILS_MATCH_RAD = 0.5  # arcsec

clean = devils_clean
c_dev   = SkyCoord(clean["RAcen"], clean["DECcen"], unit="deg")
c_khost = SkyCoord(khostovan["ra_corrected"], khostovan["dec_corrected"], unit="deg")
idx_dk, sep_dk, _ = c_dev.match_to_catalog_sky(c_khost)

matched_dk   = sep_dk.arcsec < DEVILS_MATCH_RAD
khost_flag_d = khostovan["flag"][idx_dk]
khost_z_d    = khostovan["specz"][idx_dk]

# --- Headline counts ---
n_dev      = len(clean)
n_in_khost = matched_dk.sum()
n_new      = (~matched_dk).sum()
n_hi       = (clean["DEVILS_prob"] >= 0.8).sum()
n_hi_new   = ((clean["DEVILS_prob"] >= 0.8) & ~matched_dk).sum()
n_hi_in    = ((clean["DEVILS_prob"] >= 0.8) &  matched_dk).sum()

print(f"Clean DEVILS spec-z (D10)      : {n_dev:,}")
print(f"  in Khostovan @ 1\"           : {n_in_khost:,}  ({n_in_khost/n_dev:.1%})")
print(f"  NEW (no Khostovan match)     : {n_new:,}  ({n_new/n_dev:.1%})")
print()
print(f"DEVILS prob>=0.8 (secure)      : {n_hi:,}")
print(f"  matched in Khostovan         : {n_hi_in:,}")
print(f"  NEW to Khostovan             : {n_hi_new:,}  ({n_hi_new/n_hi:.1%})")

# Khostovan flag distribution of matched DEVILS-secure pairs
m_sec = (clean["DEVILS_prob"] >= 0.8) & matched_dk
print()
print("Khostovan flag of matched DEVILS-secure pairs:")
flags, counts = np.unique(khost_flag_d[m_sec], return_counts=True)
for f, c in zip(flags, counts):
    print(f"  flag={int(f):3d}: {c:,}")

# Plot summary
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.bar(["matched", "new"], [n_in_khost, n_new],
       color=["#1f77b4", "#d62728"], edgecolor="k", lw=0.4)
ax.text(0, n_in_khost, f"{n_in_khost:,}", ha="center", va="bottom")
ax.text(1, n_new,      f"{n_new:,}",      ha="center", va="bottom")
ax.set_ylabel("N (clean DEVILS spec-z)")
ax.set_title("DEVILS DR1 D10 — present vs new in Khostovan")
ax.grid(axis="y", alpha=0.25)

ax = axes[1]
ax.bar([str(int(f)) for f in flags], counts, color="#2ca02c", edgecolor="k", lw=0.4)
for x, c in zip(range(len(flags)), counts):
    ax.text(x, c, f"{c:,}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Khostovan flag of matched entry")
ax.set_ylabel("N matched DEVILS-secure pairs")
ax.set_title("Quality of matched Khostovan entry (DEVILS_prob ≥ 0.8)")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "16_devils_vs_khost_counts.png"), dpi=160)
plt.show()


In [ ]:
# === Redshift distribution of DEVILS sources NEW to Khostovan ===
new_mask = ~matched_dk
sec_mask = (clean["DEVILS_prob"] >= 0.8)

fig, ax = plt.subplots(figsize=(11, 4.5))
bins = np.arange(0.0, 3.0, 0.02)

ax.hist(clean["DEVILS_z"][new_mask],
        bins=bins, histtype="step", lw=1.3, color="0.5",
        label=f"DEVILS new — any prob (N={new_mask.sum():,})")
ax.hist(clean["DEVILS_z"][new_mask & sec_mask],
        bins=bins, histtype="stepfilled", color="#2ca02c", alpha=0.55,
        edgecolor="#1a7a1a",
        label=f"DEVILS new — prob≥0.8 (N={(new_mask & sec_mask).sum():,})")

for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)
    ax.text(cfg["zc"], 0.97, name, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color=cfg["color"])
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)

ax.set_xlabel("DEVILS redshift")
ax.set_ylabel("N")
ax.set_title("Redshift distribution of DEVILS sources NOT in Khostovan")
ax.set_xlim(0, 2.5)
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "17_devils_new_zdist.png"), dpi=160)
plt.show()

# Tier-by-tier "what does DEVILS add?"
print("DEVILS new (prob>=0.8) per EMPOWER tier:")
for name in TIERS:
    m = new_mask & sec_mask & tier_mask(clean["DEVILS_z"], name)
    print(f"  {name:<22s} {m.sum():>5,}  NEW")
karma = (new_mask & sec_mask
         & (clean["DEVILS_z"] > KARMA_IZ_HB_HA[0])
         & (clean["DEVILS_z"] < KARMA_IZ_HB_HA[1]))
print(f"  KARMA IZ Hβ+Hα          {karma.sum():>5,}  NEW")


In [ ]:
# === Conflicting redshifts: DEVILS vs Khostovan secure (flag>=3) ===
both = matched_dk & (khost_z_d > 0) & (clean["DEVILS_z"] > 0)
zd = clean["DEVILS_z"][both]
zk = khost_z_d[both]
dz = (zd - zk) / (1.0 + zk)
flag_b = khost_flag_d[both]
prob_b = clean["DEVILS_prob"][both]

OUTLIER_THR = 0.01
n_pairs = both.sum()
n_out = (np.abs(dz) > OUTLIER_THR).sum()

print(f"Matched DEVILS↔Khostovan pairs (both z>0): {n_pairs:,}")
print(f"  median Δz/(1+z): {np.median(dz):+.4f}")
print(f"  NMAD           : {1.4826*np.median(np.abs(dz - np.median(dz))):.4f}")
print(f"  |Δz/(1+z)|>{OUTLIER_THR}: {n_out:,}  ({n_out/n_pairs:.2%})")
print()

# Breakdown by Khostovan flag (only meaningful where both Khost & DEVILS are secure)
print(f"  Outliers split by Khostovan flag (DEVILS_prob≥0.8):")
print(f"  {'flag':>6s}{'N':>10s}{'cat.':>10s}{'%':>8s}")
for fl in [4, 3, 2, 1, 0]:
    m_fl = (flag_b == fl) & (prob_b >= 0.8)
    n = m_fl.sum()
    if n == 0:
        continue
    o = (m_fl & (np.abs(dz) > OUTLIER_THR)).sum()
    print(f"  {fl:>6d}{n:>10,}{o:>10,}{o/n*100:>7.1f}%")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

ax = axes[0]
bins = np.linspace(-0.05, 0.05, 121)
ax.hist(np.clip(dz, bins[0], bins[-1]), bins=bins, color="#1f77b4",
        edgecolor="k", lw=0.3)
ax.axvline( OUTLIER_THR, ls="--", c="r", label=f"|Δz/(1+z)|={OUTLIER_THR}")
ax.axvline(-OUTLIER_THR, ls="--", c="r")
ax.set_xlabel("(z_DEVILS − z_Khost) / (1 + z_Khost)")
ax.set_ylabel("N pairs")
ax.set_title(f"DEVILS ↔ Khostovan z-agreement ({n_pairs:,} pairs)")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

ax = axes[1]
out = np.abs(dz) > OUTLIER_THR
# Colour by Khostovan flag for the outliers
ax.scatter(zk[~out], zd[~out], s=2.5, c="0.55", alpha=0.4, edgecolors="none",
           label=f"agree  ({(~out).sum():,})")
sc = ax.scatter(zk[out], zd[out], s=12, c=flag_b[out], cmap="plasma",
                vmin=0, vmax=4, edgecolors="k", lw=0.2,
                label=f"|Δz|>{OUTLIER_THR} ({out.sum():,})")
plt.colorbar(sc, ax=ax, label="Khostovan flag")
zmx = max(zk.max(), zd.max(), 4.0)
ax.plot([0, zmx], [0, zmx], ls=":", c="k", lw=1.0)
ax.set_xlabel("z (Khostovan)")
ax.set_ylabel("z (DEVILS)")
ax.set_xlim(0, zmx); ax.set_ylim(0, zmx)
ax.set_aspect("equal", adjustable="box")
ax.set_title("z_DEVILS vs z_Khost  (outliers coloured by Khost flag)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "18_devils_zagreement.png"), dpi=160)
plt.show()

# Hot list: catastrophic outliers vs Khostovan FLAG=4
ser_out = out & (flag_b == 4) & (prob_b >= 0.95)
print()
print(f"   Catastrophic outliers where BOTH are top-quality "
      f"(Khost flag=4 AND DEVILS_prob≥0.95): {ser_out.sum():,}")
if ser_out.sum():
    print("   (these are the candidates for manual review)")


## GOODS-S baseline — ASTRODEEP-GS43 (Merlin+21)

Photometric + photo-z + sparse spec-z catalogue covering CANDELS GOODS-S
(~0.28×0.30 deg, RA ≈ 53.0–53.3, Dec ≈ −28.0–−27.65).

- **gs43phot** (`astrodeep_gs43_phot.fits`) — 35,108 rows × 46 photometric columns.
- **gs43phys** (`astrodeep_gs43_phys.fits`) — 35,108 rows × 21 columns: `zBest`, the three
  photo-z estimators (`zLePhare`, `zEAzY`, `zz_phot`), `s_zBest` (rms of the three), `M_tau`,
  `M_deltau`, `SFR_tau`, `SFR_deltau`, `Qflag_tau`/`Qflag_deltau`, and `zspec_survey`
  (provenance tag; `'-'` when only photo-z is available).
- **astrodeep_gs43_specz_only.fits** — 4,899 rows: only the spec-z subset
  (`zspec_survey != '-'`). Surveys included are 3D-HST grism, VVDS, K20, VIMOS-10, VANDELS DR4,
  ACES, Kurk+12, Vanzella+08, CANDELSz7 etc. **Validated against 3,931 spec-z; predates all
  JWST spectroscopy.**

Next surveys to ingest before P117 (in priority order):
1. **JADES DR3** (D'Eugenio+25) — 2,375 robust NIRSpec/MSA z; 4,000 targets across GOODS-N+S.
2. **JADES DR4** (Curtis-Lake+ arXiv:2510.01033, Scholtz+ arXiv:2510.01034) — 3,297 robust z;
   2,545 at 1.5<z<5.7; 974 at z>4.
3. **MUSE-Wide DR1** (Urrutia+19) — 1,602 emission-line sources in 44 CDFS pointings.
4. **AMUSED / MUSE-UDF DR2** (Bacon+23) — 2,221 redshifts.
5. **VANDELS DR4** (Garilli+21) — 1,061 CDFS spectra (note: subset already in ASTRODEEP).
6. **FRESCO grism** (Oesch+23).


In [ ]:
# === Load ASTRODEEP-GS43 baseline ===
GS_DIR = os.path.join(DROPBOX_ROOT, "catalogues", "astrodeep_gs43")
gs43_phot = Table.read(os.path.join(GS_DIR, "astrodeep_gs43_phot.fits"))
gs43_phys = Table.read(os.path.join(GS_DIR, "astrodeep_gs43_phys.fits"))
gs43_specz = Table.read(os.path.join(GS_DIR, "astrodeep_gs43_specz_only.fits"))

print(f"ASTRODEEP-GS43 phot rows  : {len(gs43_phot):,}")
print(f"ASTRODEEP-GS43 phys rows  : {len(gs43_phys):,}")
print(f"  with spec-z attribution : {len(gs43_specz):,}")
print(f"  photo-z only             : {len(gs43_phys) - len(gs43_specz):,}")

# GOODS-S field centre (CANDELS GOODS-S) — distinct from COSMOS RA0/DEC0
GS_RA0, GS_DEC0 = 53.123, -27.81
print(f"\nGOODS-S centre adopted   : ({GS_RA0}, {GS_DEC0})")

# Build a single working "gs43" table joining ID + key cols of phys to phot RA/Dec
gs43 = gs43_phot["ID", "RAJ2000", "DEJ2000"].copy()
for col in ["zBest", "s_zBest", "zLePhare", "zEAzY", "zz_phot",
            "M_tau", "SFR_tau", "Qflag_tau", "zspec_survey"]:
    gs43[col] = gs43_phys[col]
# Build log M*/Msun
import numpy as _np
with _np.errstate(divide="ignore", invalid="ignore"):
    gs43["logM"] = _np.log10(_np.asarray(gs43["M_tau"]) * 1e9)
gs43["has_specz"] = _np.array([str(s).strip() != "-" for s in gs43["zspec_survey"]])

print(f"\nM* >= 1e10 M⊙ (logM>=10)  : {(gs43['logM']>=10).sum():,}")
print(f"M* >= 1e10 AND spec-z     : {((gs43['logM']>=10) & gs43['has_specz']).sum():,}")


In [ ]:
# === GOODS-S sky coverage (photo-z field + spec-z subset) ===
fig, ax = plt.subplots(figsize=(8.2, 7.8))

# All ASTRODEEP-GS43 sources
ax.scatter(gs43["RAJ2000"], gs43["DEJ2000"], s=1.8, c="0.78", alpha=0.4,
           edgecolors="none", label=f"ASTRODEEP-GS43 all (N={len(gs43):,})")

# Spec-z subset coloured by survey type
specz_only = gs43[gs43["has_specz"]]
# group MUSE-like + grism vs slit surveys
ax.scatter(specz_only["RAJ2000"], specz_only["DEJ2000"], s=6,
           c="#1f77b4", alpha=0.65, edgecolors="none",
           label=f"with spec-z attribution (N={len(specz_only):,})")

# Mass-cut overlay (M* >= 1e10 with spec-z)
sel = (gs43["logM"] >= 10) & gs43["has_specz"]
ax.scatter(gs43["RAJ2000"][sel], gs43["DEJ2000"][sel], s=14,
           facecolors="none", edgecolors="#d62728", lw=0.5,
           label=f"M*≥10¹⁰ + spec-z (N={sel.sum():,})")

ax.set_aspect("equal")
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.invert_xaxis()
ax.set_title("ASTRODEEP-GS43 sky coverage (CANDELS GOODS-S)")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "20_gs43_sky.png"), dpi=160)
plt.show()


In [ ]:
# === GOODS-S redshift distribution: photo-z field vs spec-z subset ===
fig, ax = plt.subplots(figsize=(11, 4.6))
z_all = _np.asarray(gs43["zBest"])
z_spec = _np.asarray(gs43["zBest"])[gs43["has_specz"]]
bins = _np.arange(0.0, 6.0, 0.025)

ax.hist(z_all[(z_all > 0) & _np.isfinite(z_all)], bins=bins,
        histtype="stepfilled", color="0.85", edgecolor="0.7",
        label=f"ASTRODEEP-GS43 zBest (N={(z_all>0).sum():,})")
ax.hist(z_spec[(z_spec > 0) & _np.isfinite(z_spec)], bins=bins,
        histtype="stepfilled", color="#1f77b4", alpha=0.65, edgecolor="navy",
        label=f"with spec-z attribution (N={len(z_spec):,})")

# Mass-cut spec-z
m = (gs43["logM"] >= 10) & gs43["has_specz"]
ax.hist(z_all[m], bins=bins, histtype="step", lw=1.6, color="#d62728",
        label=f"M*≥10¹⁰ + spec-z (N={m.sum():,})")

# Tier shading (note: KARMA IZ Hβ+Hα applies to GOODS-S too; same constraint)
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)
    ax.text(cfg["zc"], 0.97, name, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color=cfg["color"])
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)
# Flag the z=0.734 supercluster (Castellano+07 "K20 structure", Finoguenov+15)
ax.axvline(0.734, color="purple", lw=1.0, ls=":", alpha=0.8)
ax.text(0.734, 0.85, "z=0.734\nsupercluster", transform=ax.get_xaxis_transform(),
        ha="center", va="top", fontsize=7.5, color="purple")

ax.set_xlim(0, 5.0)
ax.set_yscale("log")
ax.set_xlabel("zBest (ASTRODEEP-GS43)")
ax.set_ylabel("N")
ax.set_title("GOODS-S redshift distribution — baseline (Khostovan equivalent for COSMOS)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "21_gs43_zhist.png"), dpi=160)
plt.show()

# Tabulate counts
print("Counts per EMPOWER tier (ASTRODEEP-GS43 zBest, includes photo-z):")
for name in TIERS:
    m_all = tier_mask(z_all, name)
    m_spec = tier_mask(z_all, name) & gs43["has_specz"]
    m_mass = m_spec & (gs43["logM"] >= 10)
    print(f"  {name:<22s} all={m_all.sum():>5,}   "
          f"spec-z={m_spec.sum():>5,}   "
          f"spec-z & M*≥10¹⁰={m_mass.sum():>5,}")
karma = (z_all > KARMA_IZ_HB_HA[0]) & (z_all < KARMA_IZ_HB_HA[1])
print(f"  KARMA IZ Hβ+Hα:  all={karma.sum():,}  spec-z={(karma & gs43['has_specz']).sum()}  "
      f"spec-z & M*≥10¹⁰={(karma & gs43['has_specz'] & (gs43['logM']>=10)).sum()}")


In [ ]:
# === Spec-z subset breakdown by survey of origin ===
import numpy as _np

specz = gs43[gs43["has_specz"]]
src_arr = _np.array([str(s).strip() for s in specz["zspec_survey"]])
src, cnt = _np.unique(src_arr, return_counts=True)
order = _np.argsort(cnt)[::-1]

# Plot top-15 surveys
fig, ax = plt.subplots(figsize=(11, 5))
xs = _np.arange(min(15, len(src)))
for i, k in enumerate(order[:15]):
    ax.barh(xs[i], cnt[k], color="#1f77b4", edgecolor="k", lw=0.3)
    ax.text(cnt[k] + 20, xs[i], f"{cnt[k]:,}", va="center", fontsize=8)
ax.set_yticks(xs)
ax.set_yticklabels([src[k] for k in order[:15]])
ax.invert_yaxis()
ax.set_xlabel("N sources in ASTRODEEP-GS43 with this attribution")
ax.set_title("ASTRODEEP-GS43 spec-z provenance — top 15 surveys")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "22_gs43_survey_breakdown.png"), dpi=160)
plt.show()

print("Full survey list (descending):")
for k in order:
    print(f"  {src[k]:<25s} {cnt[k]:,}")


In [ ]:
# === Mass / Y-band funnel (no Y_mag in phys, but M* funnel is the main filter) ===
fig, ax = plt.subplots(figsize=(8.5, 5))
m_ok = _np.isfinite(gs43["logM"]) & _np.isfinite(gs43["zBest"]) & (gs43["zBest"] > 0)
ax.scatter(gs43["zBest"][m_ok], gs43["logM"][m_ok], s=2, c="0.7", alpha=0.4,
           edgecolors="none", label=f"ASTRODEEP-GS43 all (N={m_ok.sum():,})")

mask_s = m_ok & gs43["has_specz"]
ax.scatter(gs43["zBest"][mask_s], gs43["logM"][mask_s], s=5, c="#1f77b4",
           alpha=0.7, edgecolors="none",
           label=f"with spec-z (N={mask_s.sum():,})")

# Highlight EMPOWER selection floor: logM>=10
ax.axhline(10.0, color="r", ls="--", label="log M* = 10 (EMPOWER floor)")

# Tier shading
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.5)
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)

ax.set_xlim(0, 5.0)
ax.set_ylim(6.5, 12.0)
ax.set_xlabel("zBest")
ax.set_ylabel("log M* / M⊙  (tau-SFH)")
ax.set_title("GOODS-S mass funnel for EMPOWER target selection")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "23_gs43_mass_funnel.png"), dpi=160)
plt.show()

# What fraction of mass-selected EMPOWER candidates already has a spec-z, per tier?
print("EMPOWER-selectable galaxies (M*≥10¹⁰) inside tier windows:")
for name in TIERS:
    m_tier = tier_mask(gs43["zBest"], name) & (gs43["logM"] >= 10)
    n = m_tier.sum()
    n_sp = (m_tier & gs43["has_specz"]).sum()
    frac = n_sp / max(n, 1)
    print(f"  {name:<22s} candidates={n:>5,}   with spec-z={n_sp:>5,} ({frac:.1%})  "
          f"-> NEED spec-z for {n - n_sp:,} of them")


### Adding DJA NIRSpec (≈ JADES surrogate) on top of the ASTRODEEP-GS43 baseline

The DJA NIRSpec v4.4 compilation re-reduces all public JWST NIRSpec data, including JADES,
JEMS, and the various GO/DDT programs targeting GOODS-S. Inside our GOODS-S box, **18,822 of
the 22,358 DJA entries are JADES-tagged** (root contains "jades"). So treating DJA as the
"JWST" add-on is a faithful surrogate for JADES DR3/DR4 plus the other public GOODS-S
NIRSpec programs.

**Important interpretation note:** ASTRODEEP-GS43's `zBest` column is a *photo-z* (the
median of `zLePhare`, `zEAzY`, `zz_phot`), and `zspec_survey` is just a tag flagging which
prior survey observed the source — the actual prior spec-z value is **not** carried in the
GS43 phys table. So treating GS43 as a `Khostovan-like` spec-z compilation is not exact;
the closest analogy is `Khostovan-flag≤1` (a 'this source has been observed' record).
DJA is the first true spec-z layer in this workflow for GOODS-S.

Below we follow the same baseline→add-on workflow used for COSMOS:

1. filter DJA to the GOODS-S box (RA 52.8–53.5, Dec −28.1 to −27.5);
2. crossmatch DJA → ASTRODEEP-GS43 within 1″;
3. flag sources that are *new* to the baseline (no ASTRODEEP match) and *upgrades* (ASTRODEEP
   had only a photo-z; DJA gives a secure spec-z);
4. quantify the photo-z performance on already-shared sources.


In [ ]:
# === Filter DJA NIRSpec v4.4 to the GOODS-S box ===
# Reuses `tab` loaded by the original cell 1.
assert "tab" in dir(), "Run the original DJA-load cell (`tab`) first."

GS_BOX = dict(ra_min=52.8, ra_max=53.5, dec_min=-28.1, dec_max=-27.5)
in_gs_box = ((tab["ra"]  > GS_BOX["ra_min"])  & (tab["ra"]  < GS_BOX["ra_max"])
             & (tab["dec"] > GS_BOX["dec_min"]) & (tab["dec"] < GS_BOX["dec_max"]))
dja_gs = tab[in_gs_box]
print(f"DJA inside GOODS-S box: {len(dja_gs):,}")

# Grade & program (root tag) summary
for g in [3, 2, 1, 0]:
    print(f"  grade={g}: {(dja_gs['grade'] == g).sum():,}")
print(f"  grade other/none: {(~np.isin(dja_gs['grade'], [0, 1, 2, 3])).sum():,}")

# Program breakdown
prog = np.array([str(r).split("-")[0] for r in dja_gs["root"]])
print()
print("Programs (top contributors):")
progs, cnts = np.unique(prog, return_counts=True)
for p, c in sorted(zip(progs, cnts), key=lambda x: -x[1])[:8]:
    print(f"  {p:<12s} {c:,}")

# Secure subset (grade 3, sensible z)
dja_gs_sec = dja_gs[(dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)]
print(f"\nDJA secure (grade=3 & z_best>0) inside GOODS-S box: {len(dja_gs_sec):,}")


In [ ]:
# === Crossmatch DJA-GOODS-S → ASTRODEEP-GS43 (1″ on sky) ===
c_dja  = SkyCoord(dja_gs["ra"], dja_gs["dec"], unit="deg")
c_a43  = SkyCoord(gs43_phot["RAJ2000"], gs43_phot["DEJ2000"], unit="deg")
idx_da, sep_da, _ = c_dja.match_to_catalog_sky(c_a43)
matched_da = sep_da.arcsec < 1.0

# Pull the ASTRODEEP zBest / mass / spec-z flag aligned to each DJA row
zBest_da   = gs43["zBest"][idx_da]          # baseline z (best photo/spec)
hasSp_da   = gs43["has_specz"][idx_da]      # was spec-z already in ASTRODEEP?
logM_da    = gs43["logM"][idx_da]

# Classify each DJA source against the baseline
n_dja = len(dja_gs)
status = np.full(n_dja, "", dtype=object)
new_to_base    = ~matched_da
matched_no_z   = matched_da & ~hasSp_da     # ASTRODEEP had only photo-z (UPGRADE candidates)
matched_has_z  = matched_da &  hasSp_da     # ASTRODEEP already had spec-z (CONFLICT candidates)

print(f"DJA in GOODS-S box     : {n_dja:,}")
print(f"  not in ASTRODEEP (NEW source)            : {new_to_base.sum():,}")
print(f"  matched, ASTRODEEP only photo-z (UPGRADE): {matched_no_z.sum():,}")
print(f"  matched, ASTRODEEP already has spec-z    : {matched_has_z.sum():,}")
print()

# Restrict to DJA secure (grade=3, z>0) — these are the actionable ones
sec = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
print(f"DJA secure (grade=3 & z_best>0) in GOODS-S : {sec.sum():,}")
print(f"  NEW source                               : {(sec & new_to_base ).sum():,}")
print(f"  UPGRADE (ASTRODEEP only photo-z)         : {(sec & matched_no_z).sum():,}")
print(f"  agree-test pool (ASTRODEEP had spec-z)   : {(sec & matched_has_z).sum():,}")


In [ ]:
# === Sky coverage: ASTRODEEP baseline + DJA add-on ===
fig, ax = plt.subplots(figsize=(8.5, 7.5))

# Baseline (light grey)
ax.scatter(gs43["RAJ2000"], gs43["DEJ2000"], s=1.4, c="0.85", alpha=0.45,
           edgecolors="none", label=f"ASTRODEEP-GS43 (N={len(gs43):,})")
ax.scatter(gs43["RAJ2000"][gs43["has_specz"]],
           gs43["DEJ2000"][gs43["has_specz"]],
           s=2.5, c="0.45", alpha=0.55, edgecolors="none",
           label=f"  +ASTRODEEP spec-z (N={gs43['has_specz'].sum():,})")

# DJA new vs upgrade vs duplicate
sec = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
m_new  = sec & new_to_base
m_upg  = sec & matched_no_z
m_dup  = sec & matched_has_z

ax.scatter(dja_gs["ra"][m_new], dja_gs["dec"][m_new], s=5, c="#d62728",
           alpha=0.7, edgecolors="none",
           label=f"DJA secure — NEW (N={m_new.sum():,})")
ax.scatter(dja_gs["ra"][m_upg], dja_gs["dec"][m_upg], s=5, c="#1f77b4",
           alpha=0.7, edgecolors="none",
           label=f"DJA secure — UPGRADE photo-z→spec-z (N={m_upg.sum():,})")
ax.scatter(dja_gs["ra"][m_dup], dja_gs["dec"][m_dup], s=4, c="#2ca02c",
           alpha=0.5, edgecolors="none",
           label=f"DJA secure — agree-test pool (N={m_dup.sum():,})")

ax.set_aspect("equal")
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.invert_xaxis()
ax.set_title("GOODS-S: ASTRODEEP-GS43 baseline + DJA NIRSpec add-on")
ax.legend(loc="lower left", fontsize=7.5)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "24_gs_baseline_plus_dja_sky.png"), dpi=160)
plt.show()


In [ ]:
# === Redshift distribution: where DJA fills in vs the ASTRODEEP baseline ===
fig, ax = plt.subplots(figsize=(11.5, 5))
bins = np.arange(0.0, 10.0, 0.05)

# Baseline spec-z (sources where ASTRODEEP already had a spec-z)
zB = np.asarray(gs43["zBest"])
ax.hist(zB[gs43["has_specz"] & (zB > 0)], bins=bins,
        histtype="stepfilled", color="0.75", edgecolor="0.5",
        label=f"ASTRODEEP spec-z baseline (N={(gs43['has_specz'] & (zB>0)).sum():,})")

# DJA secure subsets layered
sec = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
ax.hist(dja_gs["z_best"][sec & m_upg], bins=bins,
        histtype="stepfilled", color="#1f77b4", alpha=0.6, edgecolor="navy",
        label=f"DJA UPGRADE (was only photo-z, N={m_upg.sum():,})")
ax.hist(dja_gs["z_best"][sec & m_new], bins=bins,
        histtype="stepfilled", color="#d62728", alpha=0.55, edgecolor="darkred",
        label=f"DJA NEW (no ASTRODEEP match, N={m_new.sum():,})")

# Tier windows
for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.6)
    ax.text(cfg["zc"], 0.97, name, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color=cfg["color"])
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)
ax.axvline(0.734, color="purple", lw=1.0, ls=":", alpha=0.8)

ax.set_xlim(0, 8.0)
ax.set_yscale("log")
ax.set_xlabel("Redshift")
ax.set_ylabel("N")
ax.set_title("GOODS-S z distribution — baseline + DJA NIRSpec contribution")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "25_gs_baseline_plus_dja_zhist.png"), dpi=160)
plt.show()

# Per-tier contribution counts
print("DJA contribution per EMPOWER tier (grade=3, z_best>0):")
print(f"  {'tier':<22s}{'baseline-only':>14s}{'+UPGRADE':>10s}{'+NEW':>10s}")
for name in TIERS:
    z_dja = dja_gs["z_best"]
    m_t = tier_mask(z_dja, name) & sec
    n_base_only = int(np.sum(gs43["has_specz"] & tier_mask(zB, name) & (zB > 0)))
    n_upg = int((m_t & m_upg).sum())
    n_new = int((m_t & m_new).sum())
    print(f"  {name:<22s}{n_base_only:>14,}{n_upg:>10,}{n_new:>10,}")

# KARMA window
karma = sec & (dja_gs["z_best"] > KARMA_IZ_HB_HA[0]) & (dja_gs["z_best"] < KARMA_IZ_HB_HA[1])
print(f"\n  KARMA IZ Hβ+Hα window  : UPGRADE={(karma & m_upg).sum():,}, "
      f"NEW={(karma & m_new).sum():,}")


In [ ]:
# === DJA spec-z vs ASTRODEEP photo-z (NB: ASTRODEEP zBest is *not* a spec-z value) ===
# ASTRODEEP-GS43's `zBest` column is the best-photo-z estimate (median of zLePhare /
# zEAzY / zz_phot). `zspec_survey != "-"` only TAGS sources observed by a prior spec-z
# survey — the actual prior z value is NOT carried in the GS43 phys table.
#
# So this diagnostic is really:
#   "How well does ASTRODEEP photo-z predict the DJA NIRSpec spec-z for the same source?"
# Catastrophic disagreement here = photo-z failure, not a redshift conflict between
# spec-z surveys.

photoz_pairs = sec & matched_da & (zBest_da > 0)
zd = dja_gs["z_best"][photoz_pairs]
za = zBest_da[photoz_pairs]
dz = (zd - za) / (1.0 + za)
n_pairs = photoz_pairs.sum()
DZ_OUT = 0.05  # generous outlier cut appropriate for photo-z scatter
n_out = (np.abs(dz) > DZ_OUT).sum()

print(f"DJA-secure ↔ ASTRODEEP photo-z pairs : {n_pairs:,}")
print(f"  median Δz/(1+z)  : {np.median(dz):+.4f}")
print(f"  NMAD              : {1.4826*np.median(np.abs(dz-np.median(dz))):.4f}")
print(f"  |Δz/(1+z)|>{DZ_OUT}: {n_out:,} ({n_out/max(n_pairs,1):.1%})  ← photo-z outliers")

# Split by whether ASTRODEEP had a spec-z TAG
tag_a43 = hasSp_da[photoz_pairs]
print()
print(f"  with prior spec-z tag    (`has_specz=True`)  : {tag_a43.sum():,}")
print(f"     median Δz/(1+z): {np.median(dz[tag_a43]):+.4f}, "
      f"NMAD={1.4826*np.median(np.abs(dz[tag_a43]-np.median(dz[tag_a43]))):.4f}")
print(f"  photo-z only             (`has_specz=False`) : {(~tag_a43).sum():,}")
print(f"     median Δz/(1+z): {np.median(dz[~tag_a43]):+.4f}, "
      f"NMAD={1.4826*np.median(np.abs(dz[~tag_a43]-np.median(dz[~tag_a43]))):.4f}")

# Plot — two panels: histogram + scatter coloured by ASTRODEEP tag
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))

ax = axes[0]
bins = np.linspace(-1.5, 1.5, 121)
clipped = np.clip(dz, bins[0], bins[-1])
ax.hist(clipped[tag_a43], bins=bins, histtype="step", lw=1.3, color="#1f77b4",
        label=f"prior spec-z tag ({tag_a43.sum():,})")
ax.hist(clipped[~tag_a43], bins=bins, histtype="step", lw=1.3, color="#d62728",
        label=f"photo-z only ({(~tag_a43).sum():,})")
for v in [-DZ_OUT, DZ_OUT]:
    ax.axvline(v, ls="--", c="0.4", lw=0.7)
ax.axvline(0, ls=":", c="k", lw=0.7)
ax.set_xlabel("(z_DJA − z_ASTRODEEP_photo) / (1 + z_ASTRODEEP_photo)")
ax.set_ylabel("N")
ax.set_title(f"ASTRODEEP photo-z residuals against DJA spec-z ({n_pairs:,} pairs)")
ax.legend(fontsize=8)
ax.set_yscale("log")
ax.grid(alpha=0.25)

ax = axes[1]
# Color by tag
ax.scatter(za[~tag_a43], zd[~tag_a43], s=4, c="#d62728", alpha=0.45,
           edgecolors="none", label="photo-z only")
ax.scatter(za[tag_a43],  zd[tag_a43],  s=4, c="#1f77b4", alpha=0.45,
           edgecolors="none", label="prior spec-z tag")
zmx = float(max(zd.max(), za.max(), 4.0))
ax.plot([0, zmx], [0, zmx], ls=":", c="k", lw=1.0)
# Two zones: clipped photo-z band
ax.fill_between([0, zmx], [0, zmx*(1-DZ_OUT)-DZ_OUT],
                [0, zmx*(1+DZ_OUT)+DZ_OUT], color="0.85", alpha=0.5, zorder=0)
ax.set_xlabel("z (ASTRODEEP photo-z)")
ax.set_ylabel("z (DJA spec-z, grade=3)")
ax.set_xlim(0, zmx); ax.set_ylim(0, zmx)
ax.set_aspect("equal", adjustable="box")
ax.set_title("DJA spec-z vs ASTRODEEP photo-z")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "26_gs_photoz_vs_djaspecz.png"), dpi=160)
plt.show()

print()
print("Interpretation:")
print(f"  ASTRODEEP photo-z systematically underestimates DJA spec-z by Δz/(1+z) ≈ "
      f"{np.median(dz):+.2f}.")
print("  The largest residuals come from JADES MSA targets at z>2 where photo-z usually")
print("  fails. Tier 2/3 EMPOWER candidates selected on ASTRODEEP photo-z alone are")
print("  therefore unreliable until cross-matched against a true spec-z catalogue.")


In [ ]:
# === Combined GOODS-S spec-z budget: BEFORE vs AFTER DJA (and what's still missing) ===
# Re-derive `has_dja` for clarity (same as before, but be explicit about what's a spec-z).
c_a43_2 = SkyCoord(gs43["RAJ2000"], gs43["DEJ2000"], unit="deg")
c_dja_sec = SkyCoord(dja_gs["ra"][sec], dja_gs["dec"][sec], unit="deg")
idx_ad, sep_ad, _ = c_a43_2.match_to_catalog_sky(c_dja_sec)
has_dja = sep_ad.arcsec < 1.0    # ASTRODEEP source has DJA-secure within 1\"

# We deliberately do NOT mix `has_specz` (= prior survey tag, value unknown in this table)
# with `has_dja` (= verified DJA spec-z). They mean different things.
print("GOODS-S spec-z budget for EMPOWER candidates (M* ≥ 10¹⁰ on baseline photo-z).")
print()
print(f"  {'tier':<22s}{'cands':>8s}{'prior-tag':>11s}{'+DJA spec-z':>12s}"
      f"{'photo-z only':>14s}{'DJA-NEW':>10s}")
for name in TIERS:
    z_now = gs43["zBest"]
    m_cand = tier_mask(z_now, name) & (gs43["logM"] >= 10)
    n_cand = int(m_cand.sum())
    n_tag  = int((m_cand & gs43["has_specz"]).sum())
    n_dja  = int((m_cand & has_dja).sum())
    # Union of "any spec-z indicator" — either prior tag OR verified DJA value.
    n_with_any = int((m_cand & (gs43["has_specz"] | has_dja)).sum())
    n_photoz   = n_cand - n_with_any
    # DJA-new extras (no ASTRODEEP match) inside this tier window
    n_extras = int((sec & m_new & tier_mask(dja_gs["z_best"], name)).sum())
    print(f"  {name:<22s}{n_cand:>8,}{n_tag:>11,}{n_dja:>12,}{n_photoz:>14,}{n_extras:>10,}")

karma_m = ((gs43["zBest"] > KARMA_IZ_HB_HA[0]) & (gs43["zBest"] < KARMA_IZ_HB_HA[1])
           & (gs43["logM"] >= 10))
print(f"\n  KARMA IZ Hβ+Hα: cands={int(karma_m.sum())}  "
      f"prior-tag={int((karma_m & gs43['has_specz']).sum())}  "
      f"+DJA={int((karma_m & has_dja).sum())}  "
      f"DJA-new extras={int((sec & m_new & (dja_gs['z_best']>KARMA_IZ_HB_HA[0]) & (dja_gs['z_best']<KARMA_IZ_HB_HA[1])).sum())}")

print()
print("Caveats and next-survey priorities:")
print(" 1. ASTRODEEP-GS43 carries spec-z TAGS but not VALUES. To trust the 'prior-tag' column")
print("    as a real spec-z we need to ingest the parent catalogues (3D-HST, VANDELS DR4,")
print("    VVDS, K20, Vanzella+08, etc.).")
print(" 2. DJA already covers JADES (≈85% of DJA-GS rows). Remaining holes are at z<1 where")
print("    NIRSpec MSA programs don't target — MUSE-Wide DR1, AMUSED MUSE-UDF DR2, and")
print("    VANDELS DR4 are the priorities for filling z=0.5–2.0 in mass-selected galaxies.")
print(" 3. The 'DJA-NEW' extras (no ASTRODEEP match) typically have no UV-Y photometry in")
print("    the GS43 catalogue. Cross-match against JADES_DR2_phot / CANDELS_GS will tell")
print("    us their stellar masses before the M*≥10 cut can be applied.")


### Building a unified GOODS-S spec-z compilation

We stack four real-spec-z surveys on top of the ASTRODEEP-GS43 photo-z baseline:

| Layer | Catalogue file (under `spec_catalogues/`)                          | secure-z cut |
|-------|--------------------------------------------------------------------|--------------|
| 1     | `cosmos_khostovan25/...` — N/A (COSMOS only)                       | —            |
| 2     | `astrodeep_gs43/astrodeep_gs43_phys.fits` (baseline + spec-z tags)  | tag only     |
| 3     | DJA NIRSpec v4.4 (downloaded fresh from Zenodo, in `tab`)          | `grade == 3` |
| 4     | `muse_wide_dr1/muse_wide_dr1_emobj.fits`        (Urrutia+19)        | `Conf >= 2`  |
| 5     | `amused_museudf_dr2/amused_museudf_dr2_dr2_main.fits` (Bacon+23)   | `z > 0`      |
| 6     | `vandels_dr4_cdfs/vandels_dr4_cdfs_cdfs.fits` (Garilli+21)         | `q_zsp >= 3` |

**Dedup priority (for internal duplicates within 0.5″):**
DJA  ▶  VANDELS  ▶  MUSE-Wide  ▶  MUSE-UDF.
Rationale: JWST grating > deep optical slits > IFS broad-band — but in EMPOWER mass-selected
tier work all four are mutually consistent.


In [ ]:
# === Load MUSE-Wide DR1, AMUSED MUSE-UDF DR2, VANDELS DR4 ===
SPEC_DIR = os.path.join(DROPBOX_ROOT, "catalogues")

muse_wide  = Table.read(os.path.join(SPEC_DIR, "muse_wide_dr1",
                                     "muse_wide_dr1_emobj.fits"))
muse_udf   = Table.read(os.path.join(SPEC_DIR, "amused_museudf_dr2",
                                     "amused_museudf_dr2_dr2_main.fits"))
vandels    = Table.read(os.path.join(SPEC_DIR, "vandels_dr4_cdfs",
                                     "vandels_dr4_cdfs_cdfs.fits"))

print(f"MUSE-Wide DR1            : {len(muse_wide):,}  "
      f"(Conf>=2: {(muse_wide['Conf']>=2).sum():,})")
print(f"AMUSED MUSE-UDF DR2      : {len(muse_udf):,}  "
      f"(z>0: {(muse_udf['z']>0).sum():,})")
print(f"VANDELS DR4 CDFS         : {len(vandels):,}  "
      f"(q_zsp>=3: {(vandels['q_zsp']>=3).sum():,})")


In [ ]:
# === Build the unified gs_specz table ===
from astropy.table import vstack

def _layer(ra, dec, z, qual, survey, dat=None):
    out = Table()
    out["ra"]     = np.asarray(ra,    dtype=float)
    out["dec"]    = np.asarray(dec,   dtype=float)
    out["z"]      = np.asarray(z,     dtype=float)
    out["qual"]   = np.asarray(qual,  dtype=float)
    out["survey"] = np.array([survey]*len(out), dtype="U16")
    if dat is not None:
        for k, v in dat.items():
            out[k] = v
    return out

# DJA grade=3 sources in GOODS-S box (already filtered as `dja_gs` upstream)
sec_djasub = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
L_dja = _layer(
    dja_gs["ra"][sec_djasub], dja_gs["dec"][sec_djasub],
    dja_gs["z_best"][sec_djasub], np.full(sec_djasub.sum(), 3),
    "DJA",
)

# VANDELS q_zsp>=3
sec_van = (vandels["q_zsp"] >= 3) & (vandels["zsp"] > 0)
L_van = _layer(
    vandels["RAJ2000"][sec_van], vandels["DEJ2000"][sec_van],
    vandels["zsp"][sec_van], vandels["q_zsp"][sec_van],
    "VANDELS",
)

# MUSE-Wide Conf>=2
sec_mw = (muse_wide["Conf"] >= 2) & (muse_wide["z"] > 0)
L_mw = _layer(
    muse_wide["RAJ2000"][sec_mw], muse_wide["DEJ2000"][sec_mw],
    muse_wide["z"][sec_mw], muse_wide["Conf"][sec_mw],
    "MUSE-Wide",
)

# MUSE-UDF (DR2 main)
sec_mu = muse_udf["z"] > 0
L_mu = _layer(
    muse_udf["RAJ2000"][sec_mu], muse_udf["DEJ2000"][sec_mu],
    muse_udf["z"][sec_mu], np.full(sec_mu.sum(), 3),  # treat all as quality 3
    "MUSE-UDF",
)

# Stack in priority order
gs_specz_all = vstack([L_dja, L_van, L_mw, L_mu], join_type="outer")
print(f"All layers stacked before dedup : {len(gs_specz_all):,}")

# Internal dedup: keep first occurrence within 0.5\" of any earlier entry
from astropy.coordinates import SkyCoord
c = SkyCoord(gs_specz_all["ra"], gs_specz_all["dec"], unit="deg")
keep = np.ones(len(gs_specz_all), dtype=bool)
DEDUP_RAD = 0.5
# Walk in order; for each row i, search for *earlier* rows within 0.5\". If any, drop i.
# We use match_to_catalog_sky against the kept subset (rolling), but a fast approach:
# iterate per survey and exclude duplicates against the union of higher-priority rows.
ordered_surveys = ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]
mask_so_far = (gs_specz_all["survey"] == "DJA")
keep_arr = mask_so_far.copy()
for s in ordered_surveys[1:]:
    m_s = (gs_specz_all["survey"] == s) & ~keep_arr
    if m_s.sum() == 0:
        continue
    base_coords = SkyCoord(gs_specz_all["ra"][keep_arr],
                           gs_specz_all["dec"][keep_arr], unit="deg")
    s_coords    = SkyCoord(gs_specz_all["ra"][m_s],
                           gs_specz_all["dec"][m_s], unit="deg")
    idx, sep, _ = s_coords.match_to_catalog_sky(base_coords)
    accept = sep.arcsec >= DEDUP_RAD
    s_indices = np.where(m_s)[0]
    keep_arr[s_indices[accept]] = True

gs_specz = gs_specz_all[keep_arr]
print(f"After {DEDUP_RAD}\" dedup            : {len(gs_specz):,}")
print()
for s in ordered_surveys:
    n = (gs_specz["survey"] == s).sum()
    print(f"   {s:<12s} {n:>5,}")

# Restrict to the GOODS-S box to drop stray MUSE-Wide etc. entries (defensive)
in_box = ((gs_specz["ra"]  > 52.8) & (gs_specz["ra"]  < 53.5)
          & (gs_specz["dec"] > -28.1) & (gs_specz["dec"] < -27.5))
gs_specz = gs_specz[in_box]
print(f"\nIn GOODS-S box (final)         : {len(gs_specz):,}")


In [ ]:
# === Per-survey sky map of the unified compilation ===
fig, ax = plt.subplots(figsize=(8.2, 7.4))
ax.scatter(gs43["RAJ2000"], gs43["DEJ2000"], s=0.8, c="0.88", alpha=0.4,
           edgecolors="none", label=f"ASTRODEEP-GS43 phot (N={len(gs43):,})")

palette = {"DJA": "#1f77b4", "VANDELS": "#2ca02c",
           "MUSE-Wide": "#d62728", "MUSE-UDF": "#ff7f0e"}
sizes   = {"DJA": 3, "VANDELS": 5, "MUSE-Wide": 4, "MUSE-UDF": 4}
for s in ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]:
    m = gs_specz["survey"] == s
    ax.scatter(gs_specz["ra"][m], gs_specz["dec"][m], s=sizes[s],
               c=palette[s], alpha=0.7, edgecolors="none",
               label=f"{s} (N={m.sum():,})")

ax.set_aspect("equal")
ax.invert_xaxis()
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("Unified GOODS-S spec-z compilation — survey contributions")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "27_gs_unified_sky.png"), dpi=160)
plt.show()


In [ ]:
# === Per-survey z distribution + tier contribution table ===
fig, ax = plt.subplots(figsize=(11.5, 5))
bins = np.arange(0.0, 8.0, 0.04)
palette = {"DJA": "#1f77b4", "VANDELS": "#2ca02c",
           "MUSE-Wide": "#d62728", "MUSE-UDF": "#ff7f0e"}
for s in ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]:
    m = gs_specz["survey"] == s
    ax.hist(gs_specz["z"][m], bins=bins, histtype="step", lw=1.3,
            color=palette[s], label=f"{s} ({m.sum():,})")

for name, cfg in TIERS.items():
    ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
               color=cfg["color"], alpha=0.10)
    ax.axvline(cfg["zc"], color=cfg["color"], ls="--", lw=0.8, alpha=0.5)
    ax.text(cfg["zc"], 0.97, name, transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8, color=cfg["color"])
ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.22)
ax.axvline(0.734, color="purple", lw=1.0, ls=":", alpha=0.8)

ax.set_xlim(0, 7.0)
ax.set_yscale("log")
ax.set_xlabel("z (unified spec-z)")
ax.set_ylabel("N")
ax.set_title("GOODS-S unified spec-z — redshift distribution per layer")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "28_gs_unified_zhist.png"), dpi=160)
plt.show()

# Per-tier breakdown by survey
print(f"\n{'Tier':<22s}{'DJA':>7s}{'VANDELS':>9s}{'MUSE-Wide':>11s}{'MUSE-UDF':>10s}{'TOTAL':>8s}")
for name in TIERS:
    parts = []
    total = 0
    for s in ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]:
        n = ((gs_specz["survey"] == s) & tier_mask(gs_specz["z"], name)).sum()
        parts.append(n); total += n
    print(f"  {name:<22s}{parts[0]:>7,}{parts[1]:>9,}{parts[2]:>11,}{parts[3]:>10,}{total:>8,}")

karma = ((gs_specz["z"] > KARMA_IZ_HB_HA[0])
         & (gs_specz["z"] < KARMA_IZ_HB_HA[1]))
parts = []
for s in ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]:
    parts.append((karma & (gs_specz["survey"] == s)).sum())
print(f"  KARMA IZ Hβ+Hα window:{parts[0]:>7,}{parts[1]:>9,}{parts[2]:>11,}{parts[3]:>10,}"
      f"{sum(parts):>8,}")


In [ ]:
# === Final: EMPOWER tier candidates with VERIFIED spec-z (ASTRODEEP M*≥10¹⁰ + gs_specz) ===
# Crossmatch each ASTRODEEP source to gs_specz within 1\".
c_a43 = SkyCoord(gs43["RAJ2000"], gs43["DEJ2000"], unit="deg")
c_sp  = SkyCoord(gs_specz["ra"], gs_specz["dec"], unit="deg")
idx_as, sep_as, _ = c_a43.match_to_catalog_sky(c_sp)
matched_as = sep_as.arcsec < 1.0
sp_survey  = np.where(matched_as,
                      np.asarray(gs_specz["survey"])[idx_as.clip(0, len(gs_specz)-1)],
                      "")
sp_z       = np.where(matched_as,
                      np.asarray(gs_specz["z"])[idx_as.clip(0, len(gs_specz)-1)],
                      np.nan)

# Replace ASTRODEEP zBest with the verified spec-z when available
z_verified = np.where(matched_as, sp_z, gs43["zBest"])
# Re-apply tier+mass selection using the corrected redshift
print("EMPOWER mass-selected candidates (M* ≥ 10¹⁰) — verified spec-z per tier")
print(f"{'tier':<22s}{'cands_photoz':>14s}{'verified-z':>12s}"
      f"{'still photo-z':>15s}  surveys contributing")
for name in TIERS:
    # photo-z based "candidates" — same as before (the planning pool)
    m_cand_phot = tier_mask(gs43["zBest"], name) & (gs43["logM"] >= 10)
    # verified-z candidates (apply tier window on the corrected z)
    m_cand_ver  = tier_mask(z_verified, name) & (gs43["logM"] >= 10)
    m_verified  = m_cand_phot & matched_as
    surv_in     = sp_survey[m_verified]
    surv_str    = ", ".join(f"{s}:{(surv_in==s).sum()}" for s
                            in ["DJA","VANDELS","MUSE-Wide","MUSE-UDF"]
                            if (surv_in==s).any())
    n_phot = m_cand_phot.sum()
    n_ver  = m_verified.sum()
    print(f"  {name:<22s}{n_phot:>14,}{n_ver:>12,}{n_phot - n_ver:>15,}  {surv_str}")

# KARMA window
karma_phot = ((gs43["zBest"] > KARMA_IZ_HB_HA[0])
              & (gs43["zBest"] < KARMA_IZ_HB_HA[1])
              & (gs43["logM"] >= 10))
karma_ver  = matched_as & karma_phot
print(f"\n  KARMA IZ Hβ+Hα: cands={int(karma_phot.sum())}  "
      f"with verified spec-z={int(karma_ver.sum())}")
print(f"     surveys: " + ", ".join(
    f"{s}:{((sp_survey==s) & karma_ver).sum()}"
    for s in ["DJA","VANDELS","MUSE-Wide","MUSE-UDF"]
    if ((sp_survey==s) & karma_ver).any()))

# What about sources we add that are NEW (no ASTRODEEP photometry)?
# Project gs_specz onto ASTRODEEP and find unmatched gs_specz entries
c_sp_to_a43, sep_sp, _ = c_sp.match_to_catalog_sky(c_a43)
gs_new = sep_sp.arcsec >= 1.0
print(f"\ngs_specz sources NOT in ASTRODEEP-GS43 (no photometry yet): {gs_new.sum():,}")
print(f"   breakdown by survey:")
for s in ["DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF"]:
    print(f"      {s:<12s} {((gs_specz['survey'] == s) & gs_new).sum():>5,}")
print(f"   per tier (verified-z without mass cut):")
for name in TIERS:
    m = gs_new & tier_mask(gs_specz["z"], name)
    print(f"      {name:<22s}{int(m.sum()):>5,}")
